# 04 Synthetic Historical Weather Forecast Emulator

This notebook constructs the operational weather forecast panel for downstream ML. Horizons h=0, h=1, and h=2 use archived deterministic MEPS forecasts directly. Horizons h=3 through h=10 use a synthetic historical forecast emulator in which realised target-day weather is treated as the latent weather state and calibrated forecast error is added to create forecast-like point values and uncertainty intervals.

The medium-range extension is not an archived forecast, perfect-information benchmark, or causal estimate. It is a forecast-like scenario panel calibrated from deterministic MEPS errors, MEPS ensemble spread, historical weather distributions, and optional pressure/regime diagnostics from notebook 01d. The notebook writes `data/processed/weather_forecast_operational_windows.parquet`, `data/processed/weather_forecast_error_diagnostics_windows.parquet`, and compact audits under `outputs/synthetic_weather_model/`.


## Previous logic audit

The earlier notebook built a synthetic forecast scaffold from realised-weather history and hard-coded horizon/season RMSE curves. The current version preserves forecast-origin notation, horizon-dependent uncertainty, precipitation wet/amount modelling, cloud uncertainty, appendix tables, and diagnostics, while replacing obsolete h=1/h=2 synthetic handling and preventing realised target-day weather from entering operational ML features.


## Research-material support

Local research materials support weather forecasts as inputs to daily retail forecasting, caution against treating ex-post actual weather as operational forecast information, and motivate h=10 as the default long-horizon endpoint. Horizons h=11 to h=14 are therefore not produced by default. MET/FIMEX/NetCDF notes are used only as technical background.


## Data contract

Required processed inputs are deterministic MEPS windows from notebook 01b, realised weather windows from notebook 01a, ensemble calibration windows/audits from notebook 01c, and historical calibration windows/parameters from notebook 01d. Notebook 04 must read processed weather artifacts only, not raw FIMEX or NetCDF folders.

The operational unit is one row per `origin_date`, `origin_hour`, `origin_datetime_utc`, `target_date`, `horizon_days`, `AvdelingID`, and `aggregation_window`. The notebook produces `full_day_00_23`, `trade_08_18`, and `trade_08_19`, with `trade_08_19` as the main operational ML window. The operational output must not expose realised weather columns, forecast-error diagnostic columns, or pressure columns as ML features. Pressure/MSLP is used only internally for calibration/regime uncertainty and audit summaries.


## Emulator pipeline

The following cells regenerate the operational forecast-weather panel and associated audit outputs.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if any((candidate / marker).exists() for marker in ["README_FOR_CODEX.md", "AGENTS.md", ".git"]):
            return candidate
    raise FileNotFoundError("Could not locate project root from marker files.")


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "synthetic_weather_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DETERMINISTIC_WINDOWS_PATH = PROCESSED_DIR / "forecast_meps_daily_windows.parquet"
REALISED_WINDOWS_PATH = PROCESSED_DIR / "realised_weather_daily_windows.parquet"
HORIZON_HOUR_AUDIT_PATH = PROCESSED_DIR / "forecast_meps_horizon_hour_audit.parquet"
ENSEMBLE_WINDOWS_PATH = PROCESSED_DIR / "forecast_meps_ensemble_daily_windows.parquet"
ENSEMBLE_COVERAGE_AUDIT_PATH = PROCESSED_DIR / "forecast_meps_ensemble_coverage_audit.parquet"
HISTORICAL_CALIBRATION_WINDOWS_PATH = PROCESSED_DIR / "historical_weather_calibration_windows.parquet"
HISTORICAL_CALIBRATION_PARAMETERS_PATH = PROCESSED_DIR / "historical_weather_calibration_parameters.parquet"
OPERATIONAL_OUTPUT_PATH = PROCESSED_DIR / "weather_forecast_operational_windows.parquet"
ERROR_DIAGNOSTICS_OUTPUT_PATH = PROCESSED_DIR / "weather_forecast_error_diagnostics_windows.parquet"
SOURCE_COMPOSITION_PATH = OUTPUT_DIR / "forecast_source_composition.csv"
ERROR_DIAGNOSTICS_CSV_PATH = OUTPUT_DIR / "forecast_error_diagnostics_windows.csv"
CALIBRATION_SUMMARY_PATH = OUTPUT_DIR / "synthetic_extension_calibration_summary.csv"
CALIBRATION_WEIGHT_PATH = OUTPUT_DIR / "synthetic_extension_weights_by_horizon.csv"
COVERAGE_SUMMARY_PATH = OUTPUT_DIR / "weather_forecast_coverage_summary.csv"
OUTPUT_CHECKS_PATH = OUTPUT_DIR / "weather_forecast_operational_checks.csv"
INPUT_SCHEMA_AUDIT_PATH = OUTPUT_DIR / "weather_forecast_input_schema_audit.csv"
INTERVAL_FEATURE_SUMMARY_PATH = OUTPUT_DIR / "interval_feature_summary_by_horizon_window.csv"
BOUNDS_CHECK_SUMMARY_PATH = OUTPUT_DIR / "interval_bounds_check_summary.csv"
PRECIP_P90_SANITY_PATH = OUTPUT_DIR / "precip_p90_sanity_by_horizon_wet_prob_bucket.csv"
CLOUD_REGIME_SUMMARY_PATH = OUTPUT_DIR / "cloud_regime_probability_summary.csv"
SYNTHETIC_METHOD_SUMMARY_PATH = OUTPUT_DIR / "synthetic_emulator_method_summary.csv"
PRESSURE_REGIME_AUDIT_PATH = OUTPUT_DIR / "pressure_regime_emulator_audit.csv"
HISTORICAL_USAGE_AUDIT_PATH = OUTPUT_DIR / "historical_calibration_usage_summary.csv"
SYNTHETIC_POINT_INTERVAL_AUDIT_PATH = OUTPUT_DIR / "synthetic_emulator_point_interval_audit.csv"
CLIMATOLOGY_DRIFT_ALPHA_SCHEDULE_PATH = OUTPUT_DIR / "climatology_drift_alpha_schedule.csv"
CLIMATOLOGY_DRIFT_SOURCE_COVERAGE_PATH = OUTPUT_DIR / "climatology_drift_source_coverage.csv"

SELECTED_OPERATIONAL_WEATHER_WINDOW = "trade_08_19"
ROBUSTNESS_WEATHER_WINDOW = "trade_08_18"
BENCHMARK_WEATHER_WINDOW = "full_day_00_23"
WINDOW_ORDER = [BENCHMARK_WEATHER_WINDOW, ROBUSTNESS_WEATHER_WINDOW, SELECTED_OPERATIONAL_WEATHER_WINDOW]
PRIMARY_SUPPORTED_MAX_HORIZON = 10
INCLUDE_EXTENDED_LOW_SUPPORT_HORIZONS = False
EXTENDED_LOW_SUPPORT_MAX_HORIZON = 14
OUTPUT_HORIZONS = list(range(0, PRIMARY_SUPPORTED_MAX_HORIZON + 1))
if INCLUDE_EXTENDED_LOW_SUPPORT_HORIZONS:
    OUTPUT_HORIZONS = list(range(0, EXTENDED_LOW_SUPPORT_MAX_HORIZON + 1))
DETERMINISTIC_HORIZONS = [0, 1, 2]
SYNTHETIC_HORIZONS = [h for h in OUTPUT_HORIZONS if h >= 3]
SYNTHETIC_FORECAST_SOURCE = "synthetic_realised_anchor_error_calibrated"
WET_THRESHOLD_MM = 0.1
CLIMATOLOGY_DRIFT_ALPHA_H10 = np.float32(0.50)
CLIMATOLOGY_DRIFT_MIN_GROUP_N = 20
INTERVAL_Z_10_90 = np.float32(1.2815515655446004)
CLOUD_OPEN_THRESHOLD = np.float32(25.0)
CLOUD_OVERCAST_THRESHOLD = np.float32(75.0)
FORECAST_KEY_COLS = [
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "target_date",
    "horizon_days",
    "AvdelingID",
    "aggregation_window",
]
REAL_VALUE_MAP = {
    "temp": ("temp_fcst_mean", "temp_obs_mean"),
    "precip": ("precip_fcst_sum", "precip_obs_sum"),
    "wind": ("wind_fcst_mean", "wind_obs_mean"),
    "humid": ("humid_fcst_mean", "humid_obs_mean"),
    "cloud": ("cloud_fcst_mean", "cloud_obs_mean"),
}

# Diagnostic correlations are disabled by default after a Windows native crash in np.corrcoef.
# This affects audits only, not operational forecasts.
# Diagnostic parquet writes are optional so accepted outputs are not overwritten by default.
COMPUTE_DIAGNOSTIC_CORRELATIONS = False
DEBUG_DIAGNOSTIC_LOOP = True
WRITE_ERROR_DIAGNOSTICS_PARQUET = False


In [ ]:
def season_from_month(month: int) -> str:
    month = int(month)
    if month in (12, 1, 2):
        return "winter"
    if month in (3, 4, 5):
        return "spring"
    if month in (6, 7, 8):
        return "summer"
    return "autumn"


def add_season_month(frame: pd.DataFrame, date_col: str = "target_date") -> pd.DataFrame:
    out = frame.copy()
    out["target_month"] = pd.to_datetime(out[date_col]).dt.month.astype("int8")
    out["target_day"] = pd.to_datetime(out[date_col]).dt.day.astype("int8")
    out["season"] = out["target_month"].map(season_from_month)
    return out


def apparent_temperature(temp_c, humid_pct, wind_ms):
    temp_c = np.asarray(temp_c, dtype="float64")
    humid_pct = np.asarray(humid_pct, dtype="float64")
    wind_ms = np.asarray(wind_ms, dtype="float64")
    vapor_pressure = (humid_pct / 100.0) * 6.105 * np.exp(17.27 * temp_c / (237.7 + temp_c))
    return temp_c + 0.33 * vapor_pressure - 0.70 * wind_ms - 4.00


def rmse(x):
    x = pd.to_numeric(x, errors="coerce")
    return float(np.sqrt(np.nanmean(np.square(x)))) if x.notna().any() else np.nan


def mae(x):
    x = pd.to_numeric(x, errors="coerce")
    return float(np.nanmean(np.abs(x))) if x.notna().any() else np.nan


def corr_pair(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    mask = a.notna() & b.notna()
    if int(mask.sum()) < 3:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def horizon_progress(horizon):
    h = np.asarray(horizon, dtype="float64")
    return np.clip((h - 2.0) / max(PRIMARY_SUPPORTED_MAX_HORIZON - 2.0, 1.0), 0.0, 1.0)


# g = 2^(2.5/4.0); error grows approximately 54% per 2.5-day window.
ERROR_GROWTH_PER_2P5_DAYS = 1.54

def horizon_error_growth(horizon, variable: str):
    # Keep variable in the signature for call-site compatibility; growth is variable-independent.
    h = np.asarray(horizon, dtype="float64")
    growth = np.power(ERROR_GROWTH_PER_2P5_DAYS, (h - 2.0) / 2.5)
    return growth.astype("float32")

def uncertainty_scale_for_horizon(horizon):
    return (1.0 + 0.70 * np.power(horizon_progress(horizon), 0.85)).astype("float32")


def climatology_alpha_for_horizon(horizon, alpha_at_max: float = float(CLIMATOLOGY_DRIFT_ALPHA_H10)):
    # Baseline medium-range information decay: alpha(10)=0.50.
    # This is a modelling prior, not an archived forecast estimate.
    h = np.asarray(horizon, dtype="float64")
    horizon_span = max(float(PRIMARY_SUPPORTED_MAX_HORIZON - 2), 1.0)
    lam = -np.log(float(alpha_at_max)) / horizon_span
    alpha = np.exp(-lam * np.clip(h - 2.0, 0.0, None))
    return np.clip(alpha, float(alpha_at_max), 1.0).astype("float32")


def stable_standard_normal(frame: pd.DataFrame, variable: str) -> np.ndarray:
    key_cols = ["origin_date", "target_date", "horizon_days", "AvdelingID", "aggregation_window"]
    key = frame[key_cols].astype(str).agg("|".join, axis=1) + f"|{variable}|forecast_emulator_v1"
    hashed = pd.util.hash_pandas_object(key, index=False).astype("uint64").to_numpy(dtype="uint64")
    modulus = np.float64(4_294_967_291)
    u1 = ((hashed % np.uint64(4_294_967_291)).astype("float64") + 1.0) / (modulus + 2.0)
    u2 = (((hashed // np.uint64(4_294_967_291)) % np.uint64(4_294_967_291)).astype("float64") + 1.0) / (modulus + 2.0)
    z = np.sqrt(-2.0 * np.log(np.clip(u1, 1e-12, 1.0))) * np.cos(2.0 * np.pi * u2)
    return np.clip(z, -3.0, 3.0)


def pressure_regime_multiplier(regime, variable: str) -> np.ndarray:
    base = pd.Series(regime, dtype="object").fillna("neutral").astype(str)
    maps = {
        "temp": {"stable_high": 0.98, "neutral": 1.00, "low_unsettled": 1.03, "transition": 1.05, "unknown": 1.00},
        "humid": {"stable_high": 0.98, "neutral": 1.00, "low_unsettled": 1.04, "transition": 1.05, "unknown": 1.00},
        "wind": {"stable_high": 0.95, "neutral": 1.00, "low_unsettled": 1.08, "transition": 1.12, "unknown": 1.00},
        "cloud": {"stable_high": 0.92, "neutral": 1.00, "low_unsettled": 1.10, "transition": 1.12, "unknown": 1.00},
        "precip": {"stable_high": 0.90, "neutral": 1.00, "low_unsettled": 1.15, "transition": 1.20, "unknown": 1.00},
    }
    return np.clip(base.map(maps[variable]).fillna(1.0).astype("float64").to_numpy(), 0.85, 1.25).astype("float32")


def expand_bounds_around_anchor(lower, upper, anchor, margin):
    lo = pd.to_numeric(lower, errors="coerce").astype("float64").to_numpy()
    hi = pd.to_numeric(upper, errors="coerce").astype("float64").to_numpy()
    a = pd.to_numeric(anchor, errors="coerce").astype("float64").to_numpy()
    lo = np.where(np.isfinite(lo), lo, a - margin)
    hi = np.where(np.isfinite(hi), hi, a + margin)
    return np.minimum(lo, a - margin), np.maximum(hi, a + margin)
def grouped_iter(frame: pd.DataFrame, group_cols: list[str]):
    if group_cols:
        yield from frame.groupby(group_cols, observed=True, dropna=False)
    else:
        yield (), frame


def build_error_calibration_table(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for key, group in grouped_iter(frame, group_cols):
        if not isinstance(key, tuple):
            key = (key,)
        row = {col: val for col, val in zip(group_cols, key)}
        for variable, (fcst_col, obs_col) in REAL_VALUE_MAP.items():
            err = pd.to_numeric(group[fcst_col], errors="coerce") - pd.to_numeric(group[obs_col], errors="coerce")
            row[f"{variable}_bias_cal"] = float(err.mean()) if err.notna().any() else np.nan
            row[f"{variable}_rmse_cal"] = rmse(err)
            row[f"{variable}_mae_cal"] = mae(err)
            if variable == "precip":
                obs_wet = pd.to_numeric(group[obs_col], errors="coerce") >= WET_THRESHOLD_MM
                fcst_wet = pd.to_numeric(group[fcst_col], errors="coerce") >= WET_THRESHOLD_MM
                valid = obs_wet.notna() & fcst_wet.notna()
                obs_wet_n = int(obs_wet[valid].sum())
                obs_dry_n = int((~obs_wet[valid]).sum())
                row["precip_obs_wet_share_cal"] = float(obs_wet[valid].mean()) if valid.any() else np.nan
                row["precip_fcst_wet_share_cal"] = float(fcst_wet[valid].mean()) if valid.any() else np.nan
                row["precip_miss_rate_cal"] = (
                    float((obs_wet & ~fcst_wet & valid).sum() / obs_wet_n)
                    if obs_wet_n
                    else np.nan
                )
                row["precip_false_alarm_rate_cal"] = (
                    float((~obs_wet & fcst_wet & valid).sum() / obs_dry_n)
                    if obs_dry_n
                    else np.nan
                )
        rows.append(row)
    return pd.DataFrame(rows)


def build_ensemble_calibration_table(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    agg_map = {
        "temp_ens_std_cal": ("temp_ens_std", "mean"),
        "precip_ens_std_cal": ("precip_ens_std", "mean"),
        "precip_ens_wet_prob_cal": ("precip_ens_wet_prob", "mean"),
        "wind_ens_std_cal": ("wind_ens_std", "mean"),
        "humid_ens_std_cal": ("humid_ens_std", "mean"),
        "cloud_ens_std_cal": ("cloud_ens_std", "mean"),
        "ensemble_rows_cal": ("origin_date", "size"),
        "ensemble_min_members_cal": ("n_members_available", "min"),
    }
    if group_cols:
        out = frame.groupby(group_cols, observed=True, dropna=False).agg(**agg_map).reset_index()
    else:
        row = {}
        for out_col, (src_col, func) in agg_map.items():
            if func == "size":
                row[out_col] = len(frame)
            elif func == "min":
                row[out_col] = frame[src_col].min()
            elif func == "mean":
                row[out_col] = pd.to_numeric(frame[src_col], errors="coerce").mean()
            else:
                row[out_col] = getattr(frame[src_col], func)()
        out = pd.DataFrame([row])
    return out


def build_climatology_table(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    agg_map = {
        "temp_clim": ("temp_obs_mean", "mean"),
        "precip_clim": ("precip_obs_sum", "mean"),
        "wind_clim": ("wind_obs_mean", "mean"),
        "wind_max_clim": ("wind_obs_max", "mean"),
        "humid_clim": ("humid_obs_mean", "mean"),
        "cloud_clim": ("cloud_obs_mean", "mean"),
        "precip_wet_prob_clim": ("precip_wet_obs", "mean"),
        "precip_wet_amount_clim": ("precip_wet_amount_obs", "mean"),
        "climatology_observation_count": ("date", "size"),
    }
    if group_cols:
        out = frame.groupby(group_cols, observed=True, dropna=False).agg(**agg_map).reset_index()
    else:
        row = {}
        for out_col, (src_col, func) in agg_map.items():
            if func == "size":
                row[out_col] = len(frame)
            elif func == "mean":
                row[out_col] = pd.to_numeric(frame[src_col], errors="coerce").mean()
            else:
                row[out_col] = getattr(frame[src_col], func)()
        out = pd.DataFrame([row])
    return out


def attach_hierarchical_values(
    target: pd.DataFrame,
    specs: list[tuple[str, list[str], pd.DataFrame]],
    value_cols: list[str],
    level_col: str,
    default_values: dict[str, float] | None = None,
    default_level: str = "global_default_floor",
    attach_verbose: bool = True,
) -> pd.DataFrame:
    out = target.copy()
    for col in value_cols:
        if col not in out.columns:
            out[col] = np.nan
    out[level_col] = pd.Series(pd.NA, index=out.index, dtype="object")

    for level_name, keys, source in specs:
        if source is None or source.empty:
            continue
        available_cols = [c for c in value_cols if c in source.columns]
        if not available_cols:
            continue
        filled_at_level = pd.Series(False, index=out.index)
        if keys:
            # Require merge keys on both sides before fallback joins.
            missing_target_keys = [k for k in keys if k not in out.columns]
            missing_source_keys = [k for k in keys if k not in source.columns]
            if missing_target_keys or missing_source_keys:
                raise RuntimeError(
                    f"attach_hierarchical_values level '{level_name}': missing key columns "
                    f"(target missing {missing_target_keys}, source missing {missing_source_keys})."
                )
            # Fallback lookup must be key-unique.
            dup_keys = int(source.duplicated(keys).sum())
            lookup = source[keys + available_cols].drop_duplicates(keys)
            if int(lookup.duplicated(keys).sum()) > 0:
                raise RuntimeError(
                    f"attach_hierarchical_values level '{level_name}': lookup is not unique per keys "
                    f"{keys} even after drop_duplicates; refusing an unsafe many-to-many fill."
                )
            rows_before = len(out)
            tmp = out[keys].merge(lookup, on=keys, how="left")
            rows_after = len(tmp)
            # Key-unique fallback joins must not multiply rows.
            if rows_after > rows_before * 1.01:
                raise RuntimeError(
                    f"Potential many-to-many merge in fallback level {level_name} using keys {keys} "
                    f"(rows {rows_before} -> {rows_after})."
                )
            tmp.index = out.index
            for col in available_cols:
                mask = out[col].isna() & tmp[col].notna()
                if mask.any():
                    out.loc[mask, col] = tmp.loc[mask, col]
                    filled_at_level |= mask
            if attach_verbose:
                print(
                    f"  [{level_col}] level={level_name} keys={keys} source_rows={len(source)} "
                    f"dup_keys={dup_keys} lookup_rows={len(lookup)} rows {rows_before}->{rows_after} "
                    f"filled={int(filled_at_level.sum())}"
                )
        else:
            row = source[available_cols].dropna(how="all").head(1)
            if row.empty:
                continue
            for col in available_cols:
                val = row.iloc[0][col]
                if pd.notna(val):
                    mask = out[col].isna()
                    if mask.any():
                        out.loc[mask, col] = val
                        filled_at_level |= mask
            if attach_verbose:
                print(f"  [{level_col}] level={level_name} (global default fill) filled={int(filled_at_level.sum())}")
        level_mask = out[level_col].isna() & filled_at_level
        out.loc[level_mask, level_col] = level_name

    default_values = default_values or {}
    for col in value_cols:
        if out[col].isna().any():
            out[col] = out[col].fillna(default_values.get(col, 0.0))
    out[level_col] = out[level_col].fillna(default_level)
    return out


def clip_nonnegative(series):
    return np.clip(pd.to_numeric(series, errors="coerce").astype("float64"), 0.0, None)


def quantile_or_nan(series, q: float):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.nan
    return float(values.quantile(q))


def build_empirical_bounds_table(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for key, group in grouped_iter(frame, group_cols):
        if not isinstance(key, tuple):
            key = (key,)
        row = {col: val for col, val in zip(group_cols, key)}
        temp_low = quantile_or_nan(group["temp_obs_mean"], 0.01)
        temp_high = quantile_or_nan(group["temp_obs_mean"], 0.99)
        wind_high = quantile_or_nan(group["wind_obs_mean"], 0.995)
        precip_wet = pd.to_numeric(group["precip_obs_sum"], errors="coerce")
        precip_wet = precip_wet.loc[precip_wet >= WET_THRESHOLD_MM]
        row["temp_lower_bound"] = temp_low - 4.0 if pd.notna(temp_low) else np.nan
        row["temp_upper_bound"] = temp_high + 4.0 if pd.notna(temp_high) else np.nan
        row["wind_upper_bound"] = wind_high * 1.10 + 0.50 if pd.notna(wind_high) else np.nan
        row["precip_wet_amount_median_bound"] = quantile_or_nan(precip_wet, 0.50)
        row["precip_wet_amount_p90_bound"] = quantile_or_nan(precip_wet, 0.90)
        row["precip_wet_amount_p99_bound"] = quantile_or_nan(precip_wet, 0.99)
        row["empirical_bound_observation_count"] = int(len(group))
        row["empirical_wet_observation_count"] = int(len(precip_wet))
        rows.append(row)
    return pd.DataFrame(rows)


def approx_normal_cdf(z):
    # Dependency-light logistic approximation to the standard normal CDF.
    z = np.clip(np.asarray(z, dtype="float64"), -20.0, 20.0)
    return 1.0 / (1.0 + np.exp(-1.702 * z))


def add_bounded_symmetric_interval(
    frame: pd.DataFrame,
    mean_col: str,
    std_col: str,
    prefix: str,
    lower=None,
    upper=None,
) -> pd.DataFrame:
    out = frame.copy()
    mean = pd.to_numeric(out[mean_col], errors="coerce").astype("float64").to_numpy()
    std = np.clip(pd.to_numeric(out[std_col], errors="coerce").astype("float64").to_numpy(), 0.0, None)
    p10 = mean - float(INTERVAL_Z_10_90) * std
    p90 = mean + float(INTERVAL_Z_10_90) * std
    if lower is not None:
        lower_arr = (
            pd.to_numeric(lower, errors="coerce").astype("float64").to_numpy()
            if isinstance(lower, pd.Series)
            else np.asarray(lower, dtype="float64")
        )
        lower_arr = np.where(np.isfinite(lower_arr), lower_arr, -np.inf)
        lower_arr = np.minimum(lower_arr, mean)
        p10 = np.maximum(p10, lower_arr)
        p90 = np.maximum(p90, lower_arr)
    if upper is not None:
        upper_arr = (
            pd.to_numeric(upper, errors="coerce").astype("float64").to_numpy()
            if isinstance(upper, pd.Series)
            else np.asarray(upper, dtype="float64")
        )
        upper_arr = np.where(np.isfinite(upper_arr), upper_arr, np.inf)
        upper_arr = np.maximum(upper_arr, mean)
        p10 = np.minimum(p10, upper_arr)
        p90 = np.minimum(p90, upper_arr)
    p10 = np.minimum(p10, mean)
    p90 = np.maximum(p90, mean)
    out[f"{prefix}_p10"] = p10.astype("float32")
    out[f"{prefix}_p90"] = p90.astype("float32")
    out[f"{prefix}_interval_width"] = np.maximum(p90 - p10, 0.0).astype("float32")
    return out


def add_precipitation_interval(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    wet_prob = np.clip(
        pd.to_numeric(out["precip_wet_prob"], errors="coerce").astype("float64").to_numpy(),
        0.0,
        1.0,
    )
    precip_mean = np.clip(
        pd.to_numeric(out["precip_fcst_sum"], errors="coerce").astype("float64").to_numpy(),
        0.0,
        None,
    )
    amount_uncertainty = np.clip(
        pd.to_numeric(out["precip_amount_uncertainty"], errors="coerce").astype("float64").to_numpy(),
        0.0,
        None,
    )
    empirical_median_raw = pd.to_numeric(
        out["precip_wet_amount_median_bound"],
        errors="coerce",
    ).astype("float64").to_numpy()
    empirical_median = np.clip(
        np.where(np.isfinite(empirical_median_raw), empirical_median_raw, WET_THRESHOLD_MM),
        WET_THRESHOLD_MM,
        None,
    )
    empirical_p90_raw = pd.to_numeric(
        out["precip_wet_amount_p90_bound"],
        errors="coerce",
    ).astype("float64").to_numpy()
    empirical_p90 = np.clip(
        np.where(np.isfinite(empirical_p90_raw), empirical_p90_raw, empirical_median),
        WET_THRESHOLD_MM,
        None,
    )
    empirical_p99_raw = pd.to_numeric(
        out["precip_wet_amount_p99_bound"],
        errors="coerce",
    ).astype("float64").to_numpy()
    empirical_p99 = np.clip(
        np.where(np.isfinite(empirical_p99_raw), empirical_p99_raw, empirical_p90),
        WET_THRESHOLD_MM,
        None,
    )

    conditional_mean_from_signal = np.where(
        wet_prob > 0.02,
        precip_mean / np.maximum(wet_prob, 0.02),
        0.0,
    )
    conditional_center = np.where(
        precip_mean > 0.0,
        0.80 * conditional_mean_from_signal
        + 0.20 * np.minimum(
            empirical_median,
            np.maximum(conditional_mean_from_signal, WET_THRESHOLD_MM),
        ),
        0.0,
    )
    # Conditional wet-amount upper bound; sqrt(wet_prob) dampens intervals for dry signals.
    conditional_upper = (
        conditional_center
        + float(INTERVAL_Z_10_90)
        * amount_uncertainty
        * np.sqrt(np.clip(wet_prob, 0.0, 1.0))
    )
    empirical_cap = np.maximum.reduce([empirical_p99, empirical_p90, conditional_center, precip_mean])
    conditional_upper = np.minimum(np.maximum(conditional_upper, conditional_center), empirical_cap)
    unconditional_p90 = np.where(wet_prob <= 0.10, 0.0, conditional_upper)
    unconditional_p90 = np.where(wet_prob >= 0.50, np.maximum(unconditional_p90, precip_mean), unconditional_p90)

    out["precip_fcst_p10"] = np.zeros(len(out), dtype="float32")
    out["precip_wet_amount_p90"] = conditional_upper.astype("float32")
    out["precip_fcst_p90"] = np.clip(unconditional_p90, 0.0, None).astype("float32")
    out["precip_fcst_interval_width"] = np.maximum(
        out["precip_fcst_p90"].astype("float64")
        - out["precip_fcst_p10"].astype("float64"),
        0.0,
    ).astype("float32")
    return out


def add_cloud_regime_probabilities(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    mean = np.clip(pd.to_numeric(out["cloud_fcst_mean"], errors="coerce").astype("float64").to_numpy(), 0.0, 100.0)
    std = np.clip(pd.to_numeric(out["cloud_fcst_std"], errors="coerce").astype("float64").to_numpy(), 0.0, None)
    std_eff = np.maximum(std, 1e-3)
    open_prob = approx_normal_cdf((float(CLOUD_OPEN_THRESHOLD) - mean) / std_eff)
    overcast_prob = 1.0 - approx_normal_cdf((float(CLOUD_OVERCAST_THRESHOLD) - mean) / std_eff)
    partly_prob = np.clip(1.0 - open_prob - overcast_prob, 0.0, 1.0)
    deterministic = std < 1e-6
    if deterministic.any():
        open_prob = np.where(deterministic & (mean <= float(CLOUD_OPEN_THRESHOLD)), 1.0, open_prob)
        overcast_prob = np.where(deterministic & (mean >= float(CLOUD_OVERCAST_THRESHOLD)), 1.0, overcast_prob)
        partly_prob = np.where(
            deterministic
            & (mean > float(CLOUD_OPEN_THRESHOLD))
            & (mean < float(CLOUD_OVERCAST_THRESHOLD)),
            1.0,
            partly_prob,
        )
        open_prob = np.where(deterministic & (mean > float(CLOUD_OPEN_THRESHOLD)), 0.0, open_prob)
        overcast_prob = np.where(deterministic & (mean < float(CLOUD_OVERCAST_THRESHOLD)), 0.0, overcast_prob)
    total = np.maximum(open_prob + partly_prob + overcast_prob, 1e-12)
    out["cloud_open_prob"] = np.clip(open_prob / total, 0.0, 1.0).astype("float32")
    out["cloud_partly_prob"] = np.clip(partly_prob / total, 0.0, 1.0).astype("float32")
    out["cloud_overcast_prob"] = np.clip(overcast_prob / total, 0.0, 1.0).astype("float32")
    return out


def log_state(label: str, *frames) -> None:
    """Lightweight memory/row diagnostics for cell-by-cell debugging.

    Pass either bare DataFrames/Series or ``(name, obj)`` tuples. This helper must
    never raise: it is a debugging aid, not part of the modelling contract. psutil
    RSS is reported only if psutil is importable.
    """
    try:
        print(f"[state] {label}")
        for fr in frames:
            try:
                if isinstance(fr, tuple) and len(fr) == 2:
                    name, obj = fr
                else:
                    name, obj = None, fr
                if isinstance(obj, pd.DataFrame):
                    mem_mb = float(obj.memory_usage(deep=True).sum()) / 1e6
                    print(f"    {name or 'df'}: shape={obj.shape} mem={mem_mb:.1f} MB")
                elif isinstance(obj, pd.Series):
                    mem_mb = float(obj.memory_usage(deep=True)) / 1e6
                    print(f"    {name or 'series'}: len={len(obj)} mem={mem_mb:.1f} MB")
                else:
                    print(f"    {name or 'obj'}: {type(obj).__name__}")
            except Exception as frame_error:  # pragma: no cover - diagnostics only
                print(f"    <frame diagnostic failed: {frame_error}>")
        try:
            import os as _os

            import psutil  # type: ignore

            rss_mb = psutil.Process(_os.getpid()).memory_info().rss / 1e6
            print(f"    process RSS={rss_mb:.1f} MB")
        except Exception:
            pass
    except Exception:
        # Diagnostics must not break panel construction.
        pass


In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Selected operational weather window: {SELECTED_OPERATIONAL_WEATHER_WINDOW}")
print(f"Configured horizons: {OUTPUT_HORIZONS}")

required_inputs = {
    "deterministic_windows": DETERMINISTIC_WINDOWS_PATH,
    "realised_windows": REALISED_WINDOWS_PATH,
    "horizon_hour_audit": HORIZON_HOUR_AUDIT_PATH,
    "ensemble_windows": ENSEMBLE_WINDOWS_PATH,
    "ensemble_coverage_audit": ENSEMBLE_COVERAGE_AUDIT_PATH,
    "historical_calibration_windows": HISTORICAL_CALIBRATION_WINDOWS_PATH,
    "historical_calibration_parameters": HISTORICAL_CALIBRATION_PARAMETERS_PATH,
}
schema_rows = []
for name, path in required_inputs.items():
    if not path.exists():
        raise FileNotFoundError(f"Required input missing for {name}: {path}")
    meta = pq.ParquetFile(path)
    schema_rows.append({
        "input": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "rows": meta.metadata.num_rows,
        "columns": len(meta.schema_arrow.names),
        "size_bytes": path.stat().st_size,
        "column_names": ",".join(meta.schema_arrow.names),
    })
input_schema_audit = pd.DataFrame(schema_rows)
input_schema_audit.to_csv(INPUT_SCHEMA_AUDIT_PATH, index=False)
display(input_schema_audit[["input", "path", "rows", "columns", "size_bytes"]])


In [ ]:
# Notebook 04 reads processed inputs only, never raw FIMEX/NetCDF folders.
det_windows = pd.read_parquet(DETERMINISTIC_WINDOWS_PATH)
realised_windows = pd.read_parquet(REALISED_WINDOWS_PATH)
horizon_hour_audit = pd.read_parquet(HORIZON_HOUR_AUDIT_PATH)
ensemble_windows = pd.read_parquet(ENSEMBLE_WINDOWS_PATH)
ensemble_coverage_audit = pd.read_parquet(ENSEMBLE_COVERAGE_AUDIT_PATH)
historical_calibration_windows = pd.read_parquet(HISTORICAL_CALIBRATION_WINDOWS_PATH)
historical_calibration_parameters = pd.read_parquet(HISTORICAL_CALIBRATION_PARAMETERS_PATH)

for frame, date_cols in [
    (det_windows, ["origin_date", "origin_datetime_utc", "target_date"]),
    (realised_windows, ["date"]),
    (
        horizon_hour_audit,
        [
            "origin_date",
            "origin_datetime_utc",
            "target_date",
            "target_datetime_utc",
            "target_datetime_local",
        ],
    ),
    (ensemble_windows, ["origin_date", "origin_datetime_utc", "target_date"]),
    (historical_calibration_windows, ["date"]),
]:
    for col in date_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col])

for name, frame in [
    ("deterministic", det_windows),
    ("realised", realised_windows),
    ("ensemble", ensemble_windows),
    ("ensemble_audit", ensemble_coverage_audit),
    ("historical_calibration_windows", historical_calibration_windows),
    ("historical_calibration_parameters", historical_calibration_parameters),
]:
    print(name, frame.shape)
log_state(
    "after load processed inputs",
    ("det_windows", det_windows),
    ("realised_windows", realised_windows),
    ("ensemble_windows", ensemble_windows),
    ("historical_calibration_windows", historical_calibration_windows),
)


In [ ]:
# Deterministic forecast-error diagnostics are calibration/audit outputs only.
# They do not feed the operational forecast panel.
# Correlations are disabled by default after a Windows native crash in np.corrcoef; this affects audits only.
realised_for_merge = realised_windows.rename(columns={"date": "target_date"}).copy()
det_eval = det_windows.loc[det_windows["horizon_days"].isin([0, 1, 2])].merge(
    realised_for_merge,
    on=["target_date", "AvdelingID", "aggregation_window"],
    how="inner",
    suffixes=("", "_obsrow"),
)
det_eval = add_season_month(det_eval, "target_date")
log_state("after build det_eval (h0-h2 deterministic vs realised)", ("det_eval", det_eval))


def _safe_diagnostic_corr(fcst_arr, obs_arr):
    """Correlation for diagnostics only. Returns np.nan unless explicitly enabled.

    Disabled by default because np.corrcoef previously triggered a native kernel crash on
    Windows inside this loop. When enabled, it works on float64 arrays, drops non-finite
    pairs, requires >=3 observations and non-zero variance, and never raises.
    """
    if not COMPUTE_DIAGNOSTIC_CORRELATIONS:
        return np.nan
    try:
        f = np.asarray(fcst_arr, dtype="float64")
        o = np.asarray(obs_arr, dtype="float64")
        m = np.isfinite(f) & np.isfinite(o)
        if int(m.sum()) < 3:
            return np.nan
        fm, om = f[m], o[m]
        if float(np.std(fm)) == 0.0 or float(np.std(om)) == 0.0:
            return np.nan
        return float(np.corrcoef(fm, om)[0, 1])
    except Exception:
        return np.nan


diag_rows = []
for gid, ((window, horizon, season), group) in enumerate(
    det_eval.groupby(["aggregation_window", "horizon_days", "season"], observed=True)
):
    if DEBUG_DIAGNOSTIC_LOOP:
        print(f"[diag] group {gid}: window={window}, h={horizon}, season={season}, rows={len(group)}", flush=True)
    for variable, (fcst_col, obs_col) in REAL_VALUE_MAP.items():
        # Use local float64 arrays for crash-resistant error statistics.
        fcst = pd.to_numeric(group[fcst_col], errors="coerce").to_numpy(dtype="float64", copy=False)
        obs = pd.to_numeric(group[obs_col], errors="coerce").to_numpy(dtype="float64", copy=False)
        err = fcst - obs
        valid = np.isfinite(err)
        n = int(valid.sum())
        if DEBUG_DIAGNOSTIC_LOOP:
            print(f"[diag]   start var={variable} fcst_col={fcst_col} obs_col={obs_col} n={n}", flush=True)
        if n > 0:
            ev = err[valid]
            bias = float(ev.mean())
            rmse_v = float(np.sqrt(np.mean(ev * ev)))
            mae_v = float(np.mean(np.abs(ev)))
        else:
            bias = rmse_v = mae_v = np.nan
        row = {
            "aggregation_window": window,
            "horizon_days": int(horizon),
            "season": season,
            "variable": variable,
            "n": n,
            "bias": bias,
            "rmse": rmse_v,
            "mae": mae_v,
            "corr": _safe_diagnostic_corr(fcst, obs),
        }
        if variable == "precip":
            # Precipitation diagnostics use valid forecast-realised pairs only.
            pv = np.isfinite(obs) & np.isfinite(fcst)
            total_valid = int(pv.sum())
            ov, fv = obs[pv], fcst[pv]
            obs_wet_v = ov >= WET_THRESHOLD_MM
            fcst_wet_v = fv >= WET_THRESHOLD_MM
            obs_wet_n = int(obs_wet_v.sum())
            obs_dry_n = int((~obs_wet_v).sum())
            hits = int((obs_wet_v & fcst_wet_v).sum())
            misses = int((obs_wet_v & ~fcst_wet_v).sum())
            false_alarms = int((~obs_wet_v & fcst_wet_v).sum())
            correct = int((obs_wet_v == fcst_wet_v).sum())
            row.update({
                "obs_wet_share": float(obs_wet_v.mean()) if total_valid else np.nan,
                "fcst_wet_share": float(fcst_wet_v.mean()) if total_valid else np.nan,
                "hit_rate": float(hits / obs_wet_n) if obs_wet_n else np.nan,
                "miss_rate": float(misses / obs_wet_n) if obs_wet_n else np.nan,
                "false_alarm_rate": float(false_alarms / obs_dry_n) if obs_dry_n else np.nan,
                "accuracy": float(correct / total_valid) if total_valid else np.nan,
            })
        if DEBUG_DIAGNOSTIC_LOOP:
            print(
                f"[diag]   done  var={variable} n={n} "
                f"bias={bias:.4g} rmse={rmse_v:.4g} mae={mae_v:.4g}",
                flush=True,
            )
        diag_rows.append(row)

forecast_error_diagnostics = pd.DataFrame(diag_rows)
# CSV diagnostics are always safe; optional parquet writes must not overwrite accepted outputs by default.
forecast_error_diagnostics.to_csv(ERROR_DIAGNOSTICS_CSV_PATH, index=False)
if WRITE_ERROR_DIAGNOSTICS_PARQUET:
    try:
        forecast_error_diagnostics.to_parquet(ERROR_DIAGNOSTICS_OUTPUT_PATH, index=False)
        print(f"Wrote diagnostics parquet: {ERROR_DIAGNOSTICS_OUTPUT_PATH}", flush=True)
    except Exception as exc:
        print(
            f"WARNING: diagnostics parquet write failed ({type(exc).__name__}: {exc}); "
            "CSV still written.",
            flush=True,
        )
else:
    print(
        "Diagnostics parquet write skipped (WRITE_ERROR_DIAGNOSTICS_PARQUET=False); "
        "existing accepted parquet left untouched.",
        flush=True,
    )
print(f"Wrote diagnostics rows={len(forecast_error_diagnostics):,}", flush=True)


In [ ]:
# Hierarchical deterministic-error and ensemble-spread calibration.
# Realised weather is used only to estimate grouped calibration objects.
# Missing ensemble rows fall back to grouped deterministic-error calibration.
det_eval = det_eval.assign(calibration_horizon=det_eval["horizon_days"].astype("int16"))
ens = add_season_month(ensemble_windows, "target_date").assign(
    calibration_horizon=lambda x: x["horizon_days"].astype("int16")
)
ens = ens.loc[ens["horizon_days"].isin(DETERMINISTIC_HORIZONS)].copy()

ERROR_VALUE_COLS = []
for variable in REAL_VALUE_MAP:
    ERROR_VALUE_COLS.extend([f"{variable}_bias_cal", f"{variable}_rmse_cal", f"{variable}_mae_cal"])
ERROR_VALUE_COLS.extend([
    "precip_obs_wet_share_cal",
    "precip_fcst_wet_share_cal",
    "precip_miss_rate_cal",
    "precip_false_alarm_rate_cal",
])
ENSEMBLE_VALUE_COLS = [
    "temp_ens_std_cal",
    "precip_ens_std_cal",
    "precip_ens_wet_prob_cal",
    "wind_ens_std_cal",
    "humid_ens_std_cal",
    "cloud_ens_std_cal",
    "ensemble_rows_cal",
    "ensemble_min_members_cal",
]
CLIMATOLOGY_VALUE_COLS = [
    "temp_clim",
    "precip_clim",
    "wind_clim",
    "wind_max_clim",
    "humid_clim",
    "cloud_clim",
    "precip_wet_prob_clim",
    "precip_wet_amount_clim",
    "climatology_observation_count",
    "climatology_expected_obs",
    "climatology_min_year",
    "climatology_max_year",
    "climatology_coverage_complete",
]

ERROR_FALLBACK_SPECS = [
    (
        "store_month_horizon_window",
        ["AvdelingID", "aggregation_window", "calibration_horizon", "target_month"],
        build_error_calibration_table(
            det_eval,
            ["AvdelingID", "aggregation_window", "calibration_horizon", "target_month"],
        ),
    ),
    (
        "store_season_horizon_window",
        ["AvdelingID", "aggregation_window", "calibration_horizon", "season"],
        build_error_calibration_table(
            det_eval,
            ["AvdelingID", "aggregation_window", "calibration_horizon", "season"],
        ),
    ),
    (
        "month_horizon_window",
        ["aggregation_window", "calibration_horizon", "target_month"],
        build_error_calibration_table(
            det_eval,
            ["aggregation_window", "calibration_horizon", "target_month"],
        ),
    ),
    (
        "season_horizon_window",
        ["aggregation_window", "calibration_horizon", "season"],
        build_error_calibration_table(
            det_eval,
            ["aggregation_window", "calibration_horizon", "season"],
        ),
    ),
    (
        "horizon_window",
        ["aggregation_window", "calibration_horizon"],
        build_error_calibration_table(det_eval, ["aggregation_window", "calibration_horizon"]),
    ),
    ("horizon_global", ["calibration_horizon"], build_error_calibration_table(det_eval, ["calibration_horizon"])),
    ("variable_global", [], build_error_calibration_table(det_eval, [])),
]
ENSEMBLE_FALLBACK_SPECS = [
    (
        "store_month_horizon_window",
        ["AvdelingID", "aggregation_window", "calibration_horizon", "target_month"],
        build_ensemble_calibration_table(
            ens,
            ["AvdelingID", "aggregation_window", "calibration_horizon", "target_month"],
        ),
    ),
    (
        "store_season_horizon_window",
        ["AvdelingID", "aggregation_window", "calibration_horizon", "season"],
        build_ensemble_calibration_table(
            ens,
            ["AvdelingID", "aggregation_window", "calibration_horizon", "season"],
        ),
    ),
    (
        "month_horizon_window",
        ["aggregation_window", "calibration_horizon", "target_month"],
        build_ensemble_calibration_table(
            ens,
            ["aggregation_window", "calibration_horizon", "target_month"],
        ),
    ),
    (
        "season_horizon_window",
        ["aggregation_window", "calibration_horizon", "season"],
        build_ensemble_calibration_table(
            ens,
            ["aggregation_window", "calibration_horizon", "season"],
        ),
    ),
    (
        "horizon_window",
        ["aggregation_window", "calibration_horizon"],
        build_ensemble_calibration_table(ens, ["aggregation_window", "calibration_horizon"]),
    ),
    ("horizon_global", ["calibration_horizon"], build_ensemble_calibration_table(ens, ["calibration_horizon"])),
    ("variable_global", [], build_ensemble_calibration_table(ens, [])),
]
error_calibration_summary = ERROR_FALLBACK_SPECS[3][2].copy()
ensemble_calibration_summary = ENSEMBLE_FALLBACK_SPECS[3][2].copy()
calibration_summary = error_calibration_summary.merge(
    ensemble_calibration_summary,
    on=["aggregation_window", "calibration_horizon", "season"],
    how="outer",
)
calibration_summary.to_csv(CALIBRATION_SUMMARY_PATH, index=False)
log_state(
    "after build calibration tables",
    ("error_calibration_summary", error_calibration_summary),
    ("ensemble_calibration_summary", ensemble_calibration_summary),
    ("calibration_summary", calibration_summary),
)


In [ ]:
# For trade_08_19, prefer 01d pre-sales calibration history when store-month coverage is sufficient.
# Other windows use 01a realised windows because 01d currently covers trade_08_19 only.
hist_trade_0819_all = historical_calibration_windows.rename(columns={
    "temp_hist_mean": "temp_obs_mean", "temp_hist_min": "temp_obs_min", "temp_hist_max": "temp_obs_max",
    "precip_hist_sum": "precip_obs_sum", "wind_hist_mean": "wind_obs_mean", "wind_hist_max": "wind_obs_max",
    "humid_hist_mean": "humid_obs_mean", "cloud_hist_mean": "cloud_obs_mean",
}).copy()
hist_trade_0819_all = hist_trade_0819_all[[
    "date",
    "AvdelingID",
    "aggregation_window",
    "temp_obs_mean",
    "temp_obs_min",
    "temp_obs_max",
    "precip_obs_sum",
    "wind_obs_mean",
    "wind_obs_max",
    "humid_obs_mean",
    "cloud_obs_mean",
]]
hist_trade_0819_all["date"] = pd.to_datetime(hist_trade_0819_all["date"]).dt.normalize()
hist_trade_0819_pre_sales = hist_trade_0819_all.loc[hist_trade_0819_all["date"].le(pd.Timestamp("2021-12-31"))].copy()
pre_sales_group_n = (
    hist_trade_0819_pre_sales.assign(target_month=hist_trade_0819_pre_sales["date"].dt.month.astype("int8"))
    .groupby(["AvdelingID", "aggregation_window", "target_month"], observed=True)[
        ["temp_obs_mean", "precip_obs_sum", "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean"]
    ]
    .count()
)
pre_sales_sufficient = (not pre_sales_group_n.empty) and bool(
    pre_sales_group_n.min().min() >= int(CLIMATOLOGY_DRIFT_MIN_GROUP_N)
)
if pre_sales_sufficient:
    hist_trade_0819 = hist_trade_0819_pre_sales.copy()
    hist_trade_0819_source_label = "01d_pre_sales_2019_2021_store_month_sufficient"
else:
    hist_trade_0819 = hist_trade_0819_all.copy()
    hist_trade_0819_source_label = "01d_all_available_history_fallback"
realised_non_main_windows = realised_windows.loc[
    realised_windows["aggregation_window"].ne(SELECTED_OPERATIONAL_WEATHER_WINDOW),
    hist_trade_0819_all.columns,
].copy()
realised_clim = pd.concat([realised_non_main_windows, hist_trade_0819], ignore_index=True)
realised_clim["date"] = pd.to_datetime(realised_clim["date"])
realised_clim["target_month"] = realised_clim["date"].dt.month.astype("int8")
realised_clim["season"] = realised_clim["target_month"].map(season_from_month)
realised_clim["precip_wet_obs"] = (realised_clim["precip_obs_sum"] >= WET_THRESHOLD_MM).astype("float32")
realised_clim["precip_wet_amount_obs"] = realised_clim["precip_obs_sum"].where(realised_clim["precip_obs_sum"] >= WET_THRESHOLD_MM)


In [ ]:
# Same-calendar-day ERA5 climatology from notebook 01e is the primary h=3-h=10 centre.
# Legacy monthly/seasonal ERA5 and realised_clim remain fallback sources and are audited.
ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH = PROCESSED_DIR / "era5_longrun_same_calendar_day_climatology.parquet"
ERA5_LONGRUN_CLIMATOLOGY_PARAMETERS_PATH = PROCESSED_DIR / "era5_longrun_climatology_parameters.parquet"
ERA5_MAX_SOURCE_DATE = pd.Timestamp("2021-12-31")
ERA5_SAME_DAY_YEAR_START = 2000
ERA5_SAME_DAY_YEAR_END = 2019
era5_same_day_present = ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH.exists()
era5_climatology_present = ERA5_LONGRUN_CLIMATOLOGY_PARAMETERS_PATH.exists()
if not era5_same_day_present:
    raise FileNotFoundError(
        f"ERA5 same-calendar-day climatology not found at "
        f"{ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH}. Run notebook 01e first."
    )
if not era5_climatology_present:
    raise FileNotFoundError(
        f"ERA5 long-run climatology parameters not found at "
        f"{ERA5_LONGRUN_CLIMATOLOGY_PARAMETERS_PATH}. Run notebook 01e first."
    )
era5_same_day = pd.read_parquet(ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH)
era5_clim_params = pd.read_parquet(ERA5_LONGRUN_CLIMATOLOGY_PARAMETERS_PATH)
log_state(
    "after load ERA5 same-calendar-day + legacy parameters",
    ("era5_same_day", era5_same_day),
    ("era5_clim_params", era5_clim_params),
)
# Leakage checks: same-day uses 2000-2019; legacy parameters use <=2021-12-31.
_same_day_max_year = pd.to_numeric(era5_same_day["max_year"], errors="coerce").max()
assert _same_day_max_year <= ERA5_SAME_DAY_YEAR_END, (
    f"ERA5 same-calendar-day climatology contains years after {ERA5_SAME_DAY_YEAR_END}: max={_same_day_max_year}"
)
_max_src = pd.to_datetime(era5_clim_params["max_source_date"]).max()
assert _max_src <= ERA5_MAX_SOURCE_DATE, (
    f"ERA5 climatology contains source dates after {ERA5_MAX_SOURCE_DATE.date()}: max={_max_src}"
)

# Align ERA5 column names to the notebook 04 internal contract.
ERA5_COLUMN_RENAME = {
    "temp_clim_mean": "temp_clim",
    "wind_clim_mean": "wind_clim",
    "wind_max_clim_mean": "wind_max_clim",
    "humid_clim_mean": "humid_clim",
    "cloud_clim_mean": "cloud_clim",
    "precip_clim_mean": "precip_clim",
    "wet_day_share": "precip_wet_prob_clim",
    "precip_wet_amount_mean": "precip_wet_amount_clim",
    "n_obs": "climatology_observation_count",
    "expected_n_obs": "climatology_expected_obs",
    "min_year": "climatology_min_year",
    "max_year": "climatology_max_year",
    "coverage_complete": "climatology_coverage_complete",
    "month": "target_month",
    "day": "target_day",
}
era5_same_day_internal = era5_same_day.rename(columns=ERA5_COLUMN_RENAME).copy()
for col in ["target_month", "target_day"]:
    if col in era5_same_day_internal.columns:
        era5_same_day_internal[col] = (
            pd.to_numeric(era5_same_day_internal[col], errors="coerce")
            .astype("Int16")
            .astype("int8", errors="ignore")
        )
era5_clim_internal = era5_clim_params.rename(columns=ERA5_COLUMN_RENAME).copy()
if "target_month" in era5_clim_internal.columns:
    era5_clim_internal["target_month"] = pd.to_numeric(
        era5_clim_internal["target_month"], errors="coerce"
    ).astype("Int16").astype("int8", errors="ignore")

# Resolve the same-calendar-day climatology location key explicitly.
# Do not silently coerce a missing AvdelingID column; that would erase store-specific same-day climatology.
EXTENSION_LOCATION_KEY = "AvdelingID"  # the h=3..h=10 extension is keyed by AvdelingID


def detect_same_day_location_key(frame: pd.DataFrame) -> str:
    """Return the location-key column the same-calendar-day climatology is keyed by.

    Prefers the authoritative ``climatology_location_key_column`` declared by 01e, then
    falls back to a non-empty AvdelingID, then a non-empty store_point. Raises if neither
    is usable rather than guessing.
    """
    if "climatology_location_key_column" in frame.columns:
        declared = pd.Series(frame["climatology_location_key_column"]).dropna().unique()
        if len(declared) == 1:
            candidate = str(declared[0])
            if candidate in frame.columns and frame[candidate].notna().any():
                return candidate
    if "AvdelingID" in frame.columns and pd.to_numeric(frame["AvdelingID"], errors="coerce").notna().any():
        return "AvdelingID"
    if "store_point" in frame.columns and frame["store_point"].notna().any():
        return "store_point"
    raise KeyError(
        "ERA5 same-calendar-day climatology has no usable location key: neither a non-empty "
        "'AvdelingID' nor a non-empty 'store_point' column is present."
    )


def resolve_avdeling_store_point_mapping(frames_by_name: dict) -> "pd.DataFrame | None":
    """Build a validated one-to-one AvdelingID<->store_point mapping from already-loaded tables.

    Returns a 2-column frame ['store_point', 'AvdelingID'] or None when no loaded table carries
    both keys. Raises RuntimeError if a candidate mapping is not strictly one-to-one.
    """
    pieces, used = [], []
    for name, frame in frames_by_name.items():
        if frame is None:
            continue
        if "AvdelingID" in frame.columns and "store_point" in frame.columns:
            piece = frame[["store_point", "AvdelingID"]].dropna().drop_duplicates()
            if not piece.empty:
                pieces.append(piece)
                used.append(name)
    if not pieces:
        return None
    mapping = pd.concat(pieces, ignore_index=True).drop_duplicates()
    mapping["AvdelingID"] = pd.to_numeric(mapping["AvdelingID"], errors="coerce").astype("Int64")
    mapping["store_point"] = mapping["store_point"].astype("string")
    if mapping["store_point"].duplicated().any() or mapping["AvdelingID"].duplicated().any():
        raise RuntimeError(
            f"AvdelingID<->store_point mapping found in {used} is not one-to-one; refusing to use it."
        )
    print(f"  AvdelingID<->store_point mapping resolved from {used} ({len(mapping)} pairs).")
    return mapping


SAME_DAY_LOCATION_KEY = detect_same_day_location_key(era5_same_day_internal)
print(f"ERA5 same-day climatology location key used: {SAME_DAY_LOCATION_KEY}")

SAME_DAY_PRIMARY_KEYS = ["AvdelingID", "aggregation_window", "target_month", "target_day"]
if SAME_DAY_LOCATION_KEY == "AvdelingID":
    era5_same_day_internal["AvdelingID"] = pd.to_numeric(
        era5_same_day_internal["AvdelingID"],
        errors="coerce",
    ).astype("Int64")
    if "AvdelingID" in era5_clim_internal.columns:
        era5_clim_internal["AvdelingID"] = pd.to_numeric(
            era5_clim_internal["AvdelingID"],
            errors="coerce",
        ).astype("Int64")
    SAME_DAY_KEY_BASIS = "case A: same-calendar-day climatology already keyed by AvdelingID"
else:
    # store_point climatology requires a validated one-to-one mapping to AvdelingID.
    avd_store_map = resolve_avdeling_store_point_mapping({
        "era5_same_day": era5_same_day,
        "era5_clim_params": era5_clim_params,
        "det_windows": det_windows,
        "realised_windows": realised_windows,
        "historical_calibration_windows": historical_calibration_windows,
        "ensemble_windows": ensemble_windows,
    })
    if avd_store_map is None:
        raise RuntimeError(
            "ERA5 same-calendar-day climatology is keyed by 'store_point', but the synthetic "
            "h=3..h=10 extension is keyed by 'AvdelingID', and NO validated one-to-one "
            "AvdelingID<->store_point mapping exists in any loaded processed table "
            "(era5_longrun_same_calendar_day_climatology, era5_longrun_climatology_parameters, "
            "deterministic/realised/historical/ensemble windows). The thesis requires the "
            "same-calendar-day climatology as the PRIMARY h>=3 centre, so notebook 04 will NOT "
            "silently fall back to a non-store monthly/seasonal climatology. Re-run notebook 01e so "
            "it emits an AvdelingID-keyed same-calendar-day climatology (the 01e store-point "
            "metadata CSV currently lacks an AvdelingID column), or supply a validated mapping table."
        )
    era5_same_day_internal["store_point"] = era5_same_day_internal["store_point"].astype("string")
    era5_same_day_internal = era5_same_day_internal.merge(avd_store_map, on="store_point", how="left")
    if "store_point" in era5_clim_internal.columns:
        era5_clim_internal["store_point"] = era5_clim_internal["store_point"].astype("string")
        era5_clim_internal = era5_clim_internal.merge(avd_store_map, on="store_point", how="left")
    SAME_DAY_KEY_BASIS = "case A via validated store_point<->AvdelingID mapping"

# Validate same-calendar-day climatology before downstream construction depends on it.
_same_day_keys_present = [k for k in SAME_DAY_PRIMARY_KEYS if k in era5_same_day_internal.columns]
_same_day_complete = (
    era5_same_day_internal.loc[era5_same_day_internal[_same_day_keys_present].notna().all(axis=1)]
    if _same_day_keys_present
    else era5_same_day_internal.iloc[0:0]
)
_same_day_n_obs = pd.to_numeric(era5_same_day_internal.get("climatology_observation_count"), errors="coerce")
print("ERA5 same-calendar-day source validation:")
print(
    f"  shape={era5_same_day_internal.shape}; "
    f"location_key={SAME_DAY_LOCATION_KEY}; basis={SAME_DAY_KEY_BASIS}"
)
print(f"  primary_keys={SAME_DAY_PRIMARY_KEYS}; keys_present={_same_day_keys_present}")
if "aggregation_window" in era5_same_day_internal.columns:
    print(
        "  rows by aggregation_window: "
        f"{era5_same_day_internal['aggregation_window'].value_counts().to_dict()}"
    )
print(f"  complete-key rows={len(_same_day_complete)}")
if _same_day_keys_present:
    print(
        "  duplicate primary-key rows="
        f"{int(_same_day_complete.duplicated(SAME_DAY_PRIMARY_KEYS).sum())}"
    )
if not _same_day_n_obs.dropna().empty:
    print(
        "  climatology_observation_count min/max="
        f"({float(_same_day_n_obs.min())}, {float(_same_day_n_obs.max())})"
    )
if "climatology_coverage_complete" in era5_same_day_internal.columns:
    coverage_complete_share = float(
        pd.to_numeric(
            era5_same_day_internal["climatology_coverage_complete"],
            errors="coerce",
        ).fillna(0).mean()
    )
    print(f"  share coverage_complete={coverage_complete_share:.3f}")
log_state(
    "after resolve ERA5 same-day location key",
    ("era5_same_day_internal", era5_same_day_internal),
    ("era5_clim_internal", era5_clim_internal),
)
if len(_same_day_complete) == 0:
    raise RuntimeError(
        "ERA5 same-calendar-day climatology has zero rows with all primary keys "
        f"{SAME_DAY_PRIMARY_KEYS} populated after location-key resolution; cannot use it as the "
        "primary h>=3 centre. Investigate the location-key handling above before continuing."
    )

# era5_clim_internal must carry AvdelingID before fallback specification construction.
_required_legacy_keys = {"AvdelingID", "target_month", "aggregation_window", "season", "fallback_level"}
_missing_legacy_keys = sorted(_required_legacy_keys.difference(era5_clim_internal.columns))
if _missing_legacy_keys:
    raise RuntimeError(
        "ERA5 legacy climatology (era5_longrun_climatology_parameters) is missing required "
        f"columns {_missing_legacy_keys} after location-key resolution. The synthetic h>=3 "
        "fallback specs in the next cell require these columns. Re-run notebook 01e so both "
        "ERA5 climatology files share the same location-key contract, or fix the location-key "
        "resolution above so this column is added by merge."
    )


In [ ]:
def _era5_subset(level: int, keys: list[str]) -> pd.DataFrame:
    sub = era5_clim_internal.loc[era5_clim_internal["fallback_level"].eq(level)].copy()
    if not keys:
        return sub
    return sub.loc[sub[keys].notna().all(axis=1)]

ERA5_CLIMATOLOGY_FALLBACK_SPECS = [
    ("era5_store_same_calendar_day_window", ["AvdelingID", "aggregation_window", "target_month", "target_day"], era5_same_day_internal.loc[era5_same_day_internal[["AvdelingID", "target_month", "target_day", "aggregation_window"]].notna().all(axis=1)].copy()),
    ("era5_store_month_window",  ["AvdelingID", "aggregation_window", "target_month"], _era5_subset(1, ["AvdelingID", "target_month"])),
    ("era5_store_season_window", ["AvdelingID", "aggregation_window", "season"],       _era5_subset(2, ["AvdelingID", "season"])),
    ("era5_month_window",        ["aggregation_window", "target_month"],               _era5_subset(3, ["target_month"])),
    ("era5_season_window",       ["aggregation_window", "season"],                     _era5_subset(4, ["season"])),
    ("era5_window",              ["aggregation_window"],                               _era5_subset(5, [])),
]

# Realised-based specs remain an emergency fallback below ERA5.
REALISED_CLIMATOLOGY_FALLBACK_SPECS = [
    ("realised_store_month_window_emergency_fallback",  ["AvdelingID", "aggregation_window", "target_month"], build_climatology_table(realised_clim, ["AvdelingID", "aggregation_window", "target_month"])),
    ("realised_store_season_window_emergency_fallback", ["AvdelingID", "aggregation_window", "season"],       build_climatology_table(realised_clim, ["AvdelingID", "aggregation_window", "season"])),
    ("realised_month_window_emergency_fallback",        ["aggregation_window", "target_month"],               build_climatology_table(realised_clim, ["aggregation_window", "target_month"])),
    ("realised_season_window_emergency_fallback",       ["aggregation_window", "season"],                     build_climatology_table(realised_clim, ["aggregation_window", "season"])),
    ("realised_window_emergency_fallback",              ["aggregation_window"],                               build_climatology_table(realised_clim, ["aggregation_window"])),
    ("realised_global_emergency_fallback",              [],                                                   build_climatology_table(realised_clim, [])),
]

CLIMATOLOGY_FALLBACK_SPECS = ERA5_CLIMATOLOGY_FALLBACK_SPECS + REALISED_CLIMATOLOGY_FALLBACK_SPECS
print("Climatology fallback specs (level -> keys -> source rows):")
for _spec_name, _spec_keys, _spec_source in CLIMATOLOGY_FALLBACK_SPECS:
    print(f"  {_spec_name}: keys={_spec_keys} rows={0 if _spec_source is None else len(_spec_source)}")
log_state(
    "after build climatology fallback specs",
    ("era5_same_day_spec", ERA5_CLIMATOLOGY_FALLBACK_SPECS[0][2]),
    ("realised_global_emergency_spec", REALISED_CLIMATOLOGY_FALLBACK_SPECS[-1][2]),
)
pressure_regime_for_extension = historical_calibration_windows[["date", "AvdelingID", "aggregation_window", "pressure_regime", "pressure_tendency_regime", "pressure_anomaly_hpa"]].rename(columns={"date": "target_date"}).copy()
CALIBRATION_DEFAULTS = {col: 0.0 for col in ERROR_VALUE_COLS}
for variable in REAL_VALUE_MAP:
    CALIBRATION_DEFAULTS[f"{variable}_rmse_cal"] = 0.01
    CALIBRATION_DEFAULTS[f"{variable}_mae_cal"] = 0.01
CALIBRATION_DEFAULTS.update({"precip_fcst_wet_share_cal": 0.0, "precip_obs_wet_share_cal": 0.0})
ENSEMBLE_DEFAULTS = {
    "temp_ens_std_cal": 0.0, "precip_ens_std_cal": 0.0, "precip_ens_wet_prob_cal": np.nan,
    "wind_ens_std_cal": 0.0, "humid_ens_std_cal": 0.0, "cloud_ens_std_cal": 0.0,
    "ensemble_rows_cal": 0.0, "ensemble_min_members_cal": 0.0,
}
CLIMATOLOGY_DEFAULTS = {col: 0.0 for col in CLIMATOLOGY_VALUE_COLS}
CLIMATOLOGY_DEFAULTS.update({"humid_clim": 50.0, "cloud_clim": 50.0, "climatology_observation_count": 0.0, "climatology_expected_obs": 20.0, "climatology_min_year": np.nan, "climatology_max_year": np.nan, "climatology_coverage_complete": 0.0})

BOUNDS_VALUE_COLS = [
    "temp_lower_bound", "temp_upper_bound", "wind_upper_bound",
    "precip_wet_amount_median_bound", "precip_wet_amount_p90_bound", "precip_wet_amount_p99_bound",
    "empirical_bound_observation_count", "empirical_wet_observation_count",
]
BOUNDS_FALLBACK_SPECS = [
    ("store_month_window", ["AvdelingID", "aggregation_window", "target_month"], build_empirical_bounds_table(realised_clim, ["AvdelingID", "aggregation_window", "target_month"])),
    ("store_season_window", ["AvdelingID", "aggregation_window", "season"], build_empirical_bounds_table(realised_clim, ["AvdelingID", "aggregation_window", "season"])),
    ("month_window", ["aggregation_window", "target_month"], build_empirical_bounds_table(realised_clim, ["aggregation_window", "target_month"])),
    ("season_window", ["aggregation_window", "season"], build_empirical_bounds_table(realised_clim, ["aggregation_window", "season"])),
    ("window", ["aggregation_window"], build_empirical_bounds_table(realised_clim, ["aggregation_window"])),
    ("global", [], build_empirical_bounds_table(realised_clim, [])),
]
global_bounds = BOUNDS_FALLBACK_SPECS[-1][2]

def _global_bound_default(column: str, fallback: float) -> float:
    if global_bounds.empty or column not in global_bounds.columns:
        return fallback
    value = global_bounds.iloc[0][column]
    return float(value) if pd.notna(value) else fallback

BOUNDS_DEFAULTS = {
    "temp_lower_bound": _global_bound_default("temp_lower_bound", -35.0),
    "temp_upper_bound": _global_bound_default("temp_upper_bound", 35.0),
    "wind_upper_bound": _global_bound_default("wind_upper_bound", 35.0),
    "precip_wet_amount_median_bound": _global_bound_default("precip_wet_amount_median_bound", WET_THRESHOLD_MM),
    "precip_wet_amount_p90_bound": _global_bound_default("precip_wet_amount_p90_bound", 5.0),
    "precip_wet_amount_p99_bound": _global_bound_default("precip_wet_amount_p99_bound", 25.0),
    "empirical_bound_observation_count": _global_bound_default("empirical_bound_observation_count", 0.0),
    "empirical_wet_observation_count": _global_bound_default("empirical_wet_observation_count", 0.0),
}


def attach_calibration(frame: pd.DataFrame) -> pd.DataFrame:
    out = attach_hierarchical_values(frame, ERROR_FALLBACK_SPECS, ERROR_VALUE_COLS, "calibration_fallback_level", CALIBRATION_DEFAULTS)
    out = attach_hierarchical_values(out, ENSEMBLE_FALLBACK_SPECS, ENSEMBLE_VALUE_COLS, "uncertainty_fallback_level", ENSEMBLE_DEFAULTS, default_level="deterministic_error_no_ensemble")
    for variable in ["temp", "precip", "wind", "humid", "cloud"]:
        ens_col = f"{variable}_ens_std_cal"
        rmse_col = f"{variable}_rmse_cal"
        if ens_col in out.columns and rmse_col in out.columns:
            out[ens_col] = out[ens_col].fillna(out[rmse_col]).fillna(0.0)
    out["precip_ens_wet_prob_cal"] = out["precip_ens_wet_prob_cal"].fillna(out["precip_fcst_wet_share_cal"]).fillna(0.0)
    return out


In [ ]:
# h=0/h=1/h=2 point forecasts are preserved directly from deterministic MEPS.
det_operational = det_windows.loc[
    det_windows["horizon_days"].isin(DETERMINISTIC_HORIZONS)
    & det_windows["aggregation_window"].isin(WINDOW_ORDER)
].copy()
det_operational = add_season_month(det_operational, "target_date")
det_operational["calibration_horizon"] = det_operational["horizon_days"].astype("int16")
det_operational = attach_calibration(det_operational)
det_operational["forecast_source"] = "deterministic_meps"
det_operational["forecast_support_level"] = "observed_forecast_available"
det_operational["anchor_horizon"] = det_operational["horizon_days"].astype("int8")
det_operational["max_supported_horizon"] = np.int8(PRIMARY_SUPPORTED_MAX_HORIZON)
det_operational["h2_anchor_weight"] = np.float32(1.0)
det_operational["climatology_weight"] = np.float32(0.0)
det_operational["uncertainty_scale"] = np.float32(1.0)
det_operational["calibration_source"] = "hierarchical_deterministic_error_plus_ensemble_fallback"
det_operational["uncertainty_source"] = "hierarchical_ensemble_spread_or_deterministic_error_fallback"
det_operational["climatology_fallback_level"] = "not_used_deterministic_meps"
det_operational["climatology_source"] = "not_used_deterministic_meps"
det_operational["synthetic_anchor_type"] = "actual_deterministic_meps"
det_operational["synthetic_error_method"] = "none_actual_forecast"
det_operational["climatology_observation_count"] = 0.0
det_operational["climatology_n_obs"] = 0.0
det_operational["climatology_expected_obs"] = np.nan
det_operational["climatology_min_year"] = np.nan
det_operational["climatology_max_year"] = np.nan
det_operational["climatology_year_span"] = ""
det_operational["climatology_coverage_complete"] = np.nan
det_operational["temp_fcst_std"] = np.maximum(
    det_operational["temp_rmse_cal"],
    det_operational["temp_ens_std_cal"],
).astype("float32")
det_operational["wind_fcst_std"] = np.maximum(
    det_operational["wind_rmse_cal"],
    det_operational["wind_ens_std_cal"],
).astype("float32")
det_operational["humid_fcst_std"] = np.maximum(
    det_operational["humid_rmse_cal"],
    det_operational["humid_ens_std_cal"],
).astype("float32")
det_operational["cloud_fcst_std"] = np.maximum(
    det_operational["cloud_rmse_cal"],
    det_operational["cloud_ens_std_cal"],
).astype("float32")
det_operational["precip_amount_uncertainty"] = np.maximum(
    det_operational["precip_rmse_cal"],
    det_operational["precip_ens_std_cal"],
).astype("float32")
det_wet = (det_operational["precip_fcst_sum"] >= WET_THRESHOLD_MM).astype("float32")
det_operational["precip_wet_prob"] = np.clip(
    0.5 * det_wet + 0.5 * det_operational["precip_ens_wet_prob_cal"].fillna(det_wet),
    0.0,
    1.0,
).astype("float32")
det_operational["apparent_temp_fcst_mean"] = apparent_temperature(
    det_operational["temp_fcst_mean"],
    det_operational["humid_fcst_mean"],
    det_operational["wind_fcst_mean"],
).astype("float32")
det_operational["temp_h2_anchor_weight"] = np.float32(1.0)
det_operational["wind_h2_anchor_weight"] = np.float32(1.0)
det_operational["humid_h2_anchor_weight"] = np.float32(1.0)
det_operational["cloud_h2_anchor_weight"] = np.float32(1.0)
det_operational["precip_h2_anchor_weight"] = np.float32(1.0)


In [ ]:
# h=3-h=10 emulator: realised weather is the latent state.
# The forecast centre drifts toward local climatology with seeded calibrated error.
base_origins = det_windows.loc[
    det_windows["horizon_days"].eq(2)
    & det_windows["aggregation_window"].isin(WINDOW_ORDER)
].copy()
base_origins = base_origins[[
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "AvdelingID",
    "aggregation_window",
]].drop_duplicates()
extension = base_origins.merge(
    pd.DataFrame({"horizon_days": np.array(SYNTHETIC_HORIZONS, dtype="int8")}),
    how="cross",
)
extension["target_date"] = extension["origin_date"] + pd.to_timedelta(
    extension["horizon_days"].astype("int16"),
    unit="D",
)
log_state("after create extension (origins x synthetic horizons)", ("extension", extension))
truth_cols = [
    "target_date",
    "AvdelingID",
    "aggregation_window",
    "temp_obs_mean",
    "temp_obs_min",
    "temp_obs_max",
    "precip_obs_sum",
    "wind_obs_mean",
    "wind_obs_max",
    "humid_obs_mean",
    "cloud_obs_mean",
]
realised_truth = realised_windows.rename(columns={"date": "target_date"})[truth_cols].copy()
_rows_before_truth = len(extension)
extension = extension.merge(realised_truth, on=["target_date", "AvdelingID", "aggregation_window"], how="left")
if len(extension) > _rows_before_truth * 1.01:
    raise RuntimeError(
        f"Merging realised_truth multiplied extension rows ({_rows_before_truth} -> {len(extension)}); "
        "realised_truth is not unique on (target_date, AvdelingID, aggregation_window)."
    )
missing_truth_rows = int(extension["temp_obs_mean"].isna().sum())
extension = extension.loc[extension["temp_obs_mean"].notna()].copy()
log_state("after merge realised_truth into extension", ("extension", extension))
extension = add_season_month(extension, "target_date")
extension["calibration_horizon"] = np.int16(2)
extension = attach_calibration(extension)


In [ ]:
# Convert climatology coverage flag to numeric 0/1 before hierarchical attach.
for level_name, keys, source in CLIMATOLOGY_FALLBACK_SPECS:
    if source is not None and "climatology_coverage_complete" in source.columns:
        source["climatology_coverage_complete"] = (
            source["climatology_coverage_complete"]
            .map({True: 1.0, False: 0.0})
            .fillna(0.0)
            .astype("float32")
        )

In [ ]:
extension = attach_hierarchical_values(
    extension,
    CLIMATOLOGY_FALLBACK_SPECS,
    CLIMATOLOGY_VALUE_COLS,
    "climatology_fallback_level",
    CLIMATOLOGY_DEFAULTS,
)
extension["climatology_source"] = extension["climatology_fallback_level"].astype("object")
extension["climatology_n_obs"] = pd.to_numeric(extension["climatology_observation_count"], errors="coerce")
extension["climatology_year_span"] = np.where(
    pd.to_numeric(extension["climatology_min_year"], errors="coerce").notna()
    & pd.to_numeric(extension["climatology_max_year"], errors="coerce").notna(),
    pd.to_numeric(extension["climatology_min_year"], errors="coerce").astype("Int64").astype(str)
    + "-"
    + pd.to_numeric(extension["climatology_max_year"], errors="coerce").astype("Int64").astype(str),
    "",
)
extension = attach_hierarchical_values(
    extension,
    BOUNDS_FALLBACK_SPECS,
    BOUNDS_VALUE_COLS,
    "interval_bound_fallback_level",
    BOUNDS_DEFAULTS,
)
extension = extension.merge(
    pressure_regime_for_extension,
    on=["target_date", "AvdelingID", "aggregation_window"],
    how="left",
)
extension["pressure_regime_internal"] = extension["pressure_regime"].fillna("neutral")
extension["pressure_tendency_regime_internal"] = extension["pressure_tendency_regime"].fillna("unknown")
extension["precip_wet_amount_clim"] = extension["precip_wet_amount_clim"].fillna(extension["precip_clim"]).fillna(0.0)
print("Extension climatology fallback levels (h=3..h=10):")
print(extension["climatology_fallback_level"].value_counts().to_dict())
log_state("after attach calibration + climatology to extension", ("extension", extension))


In [ ]:
h = extension["horizon_days"]
alpha_h = climatology_alpha_for_horizon(h).astype("float64")
for col in [
    "h2_anchor_weight",
    "climatology_weight",
    "temp_h2_anchor_weight",
    "wind_h2_anchor_weight",
    "humid_h2_anchor_weight",
    "cloud_h2_anchor_weight",
    "precip_h2_anchor_weight",
]:
    extension[col] = np.float32(0.0)
extension["climatology_weight"] = (1.0 - alpha_h).astype("float32")
extension["uncertainty_scale"] = uncertainty_scale_for_horizon(h)
climatology_center_cols = {
    "temp": "temp_clim",
    "wind": "wind_clim",
    "humid": "humid_clim",
    "cloud": "cloud_clim",
}
for variable, obs_col in {
    "temp": "temp_obs_mean",
    "wind": "wind_obs_mean",
    "humid": "humid_obs_mean",
    "cloud": "cloud_obs_mean",
}.items():
    bias = pd.to_numeric(extension[f"{variable}_bias_cal"], errors="coerce").fillna(0.0).astype("float64").to_numpy()
    rmse_base = np.maximum(pd.to_numeric(extension[f"{variable}_rmse_cal"], errors="coerce").fillna(0.01).astype("float64").to_numpy(), pd.to_numeric(extension[f"{variable}_ens_std_cal"], errors="coerce").fillna(0.0).astype("float64").to_numpy())
    sigma = np.maximum(rmse_base * horizon_error_growth(h, variable).astype("float64") * pressure_regime_multiplier(extension["pressure_regime_internal"], variable).astype("float64"), 0.01)
    truth_value = pd.to_numeric(extension[obs_col], errors="coerce").astype("float64").to_numpy()
    climatology_value = pd.to_numeric(extension[climatology_center_cols[variable]], errors="coerce").fillna(pd.Series(truth_value, index=extension.index)).astype("float64").to_numpy()
    center = alpha_h * truth_value + (1.0 - alpha_h) * climatology_value
    extension[f"{variable}_synthetic_center_internal"] = center.astype("float32")
    point = center + bias + stable_standard_normal(extension, variable) * sigma
    if variable == "temp":
        lo, hi = expand_bounds_around_anchor(extension["temp_lower_bound"], extension["temp_upper_bound"], extension["temp_synthetic_center_internal"], 3.0)
        extension["temp_fcst_mean"] = np.clip(point, lo, hi).astype("float32")
        extension["temp_fcst_std"] = sigma.astype("float32")
    elif variable == "wind":
        hi = np.maximum.reduce([pd.to_numeric(extension["wind_upper_bound"], errors="coerce").fillna(35.0).astype("float64").to_numpy(), center + 2.0 * sigma, np.full(len(extension), 5.0)])
        extension["wind_fcst_mean"] = np.clip(point, 0.0, hi).astype("float32")
        wind_max_truth = pd.to_numeric(extension["wind_obs_max"], errors="coerce").astype("float64").to_numpy()
        wind_max_clim = pd.to_numeric(extension["wind_max_clim"], errors="coerce").fillna(pd.Series(wind_max_truth, index=extension.index)).astype("float64").to_numpy()
        wind_max_center = alpha_h * wind_max_truth + (1.0 - alpha_h) * wind_max_clim
        wind_max_point = wind_max_center + bias + stable_standard_normal(extension, "wind_max") * sigma
        extension["wind_fcst_max"] = np.maximum(extension["wind_fcst_mean"].astype("float64"), np.clip(wind_max_point, 0.0, hi + sigma)).astype("float32")
        extension["wind_fcst_std"] = sigma.astype("float32")
    elif variable == "humid":
        extension["humid_fcst_mean"] = np.clip(point, 0.0, 100.0).astype("float32")
        extension["humid_fcst_std"] = sigma.astype("float32")
    elif variable == "cloud":
        extension["cloud_fcst_mean"] = np.clip(point, 0.0, 100.0).astype("float32")
        extension["cloud_fcst_std"] = sigma.astype("float32")
precip_truth = pd.to_numeric(extension["precip_obs_sum"], errors="coerce").fillna(0.0).astype("float64").to_numpy()
precip_clim = pd.to_numeric(extension["precip_clim"], errors="coerce").fillna(0.0).astype("float64").to_numpy()
precip_center = np.clip(alpha_h * precip_truth + (1.0 - alpha_h) * precip_clim, 0.0, None)
extension["precip_synthetic_center_internal"] = precip_center.astype("float32")
truth_wet = (precip_truth >= WET_THRESHOLD_MM).astype("float64")
precip_sigma = np.maximum(np.maximum(pd.to_numeric(extension["precip_rmse_cal"], errors="coerce").fillna(0.01).astype("float64").to_numpy(), pd.to_numeric(extension["precip_ens_std_cal"], errors="coerce").fillna(0.0).astype("float64").to_numpy()) * horizon_error_growth(h, "precip").astype("float64") * pressure_regime_multiplier(extension["pressure_regime_internal"], "precip").astype("float64"), 0.01)
clim_wet = np.clip(pd.to_numeric(extension["precip_wet_prob_clim"], errors="coerce").fillna(0.0).astype("float64").to_numpy(), 0.0, 1.0)
latent_wet_signal = np.clip(alpha_h * truth_wet + (1.0 - alpha_h) * clim_wet, 0.0, 1.0)
ens_wet = np.clip(pd.to_numeric(extension["precip_ens_wet_prob_cal"], errors="coerce").fillna(pd.Series(clim_wet, index=extension.index)).astype("float64").to_numpy(), 0.0, 1.0)
progress = horizon_progress(h)
tw = 0.72 - 0.18 * progress
cw = 0.18 + 0.08 * progress
ew = np.clip(1.0 - tw - cw, 0.05, 0.25)
pressure_shift = pd.Series(extension["pressure_regime_internal"], dtype="object").map({"stable_high": -0.03, "neutral": 0.0, "low_unsettled": 0.04, "transition": 0.05, "unknown": 0.0}).fillna(0.0).to_numpy()
extension["precip_wet_prob"] = np.clip(tw * latent_wet_signal + cw * clim_wet + ew * ens_wet + stable_standard_normal(extension, "precip_wet") * 0.045 * pressure_regime_multiplier(extension["pressure_regime_internal"], "precip").astype("float64") + pressure_shift, 0.0, 1.0).astype("float32")
amount_bias = pd.to_numeric(extension["precip_bias_cal"], errors="coerce").fillna(0.0).astype("float64").to_numpy()
amount_noise = stable_standard_normal(extension, "precip_amount")
empirical_median = np.clip(pd.to_numeric(extension["precip_wet_amount_median_bound"], errors="coerce").fillna(WET_THRESHOLD_MM).astype("float64").to_numpy(), WET_THRESHOLD_MM, None)
false_alarm_amount = empirical_median * np.maximum(extension["precip_wet_prob"].astype("float64"), 0.02)
conditional_amount = np.where((truth_wet > 0) | (precip_center >= WET_THRESHOLD_MM), np.maximum(precip_center + amount_bias + amount_noise * precip_sigma, WET_THRESHOLD_MM), np.maximum(false_alarm_amount + np.maximum(amount_noise, 0.0) * precip_sigma * 0.20, 0.0))
empirical_p99 = np.clip(pd.to_numeric(extension["precip_wet_amount_p99_bound"], errors="coerce").fillna(25.0).astype("float64").to_numpy(), WET_THRESHOLD_MM, None)
heavy_day_cap = np.maximum.reduce([empirical_p99, precip_truth * 1.75 + 2.0 * precip_sigma, precip_center * 1.75 + 2.0 * precip_sigma])
conditional_amount = np.minimum(conditional_amount, heavy_day_cap)
extension["precip_fcst_sum"] = np.clip(extension["precip_wet_prob"].astype("float64") * conditional_amount, 0.0, None).astype("float32")
extension["precip_amount_uncertainty"] = precip_sigma.astype("float32")
extension["precip_wet_amount_p90_bound"] = np.maximum(pd.to_numeric(extension["precip_wet_amount_p90_bound"], errors="coerce").fillna(WET_THRESHOLD_MM).astype("float64"), np.maximum(precip_truth, precip_center) * 1.20 + precip_sigma).astype("float32")
extension["precip_wet_amount_p99_bound"] = np.maximum(pd.to_numeric(extension["precip_wet_amount_p99_bound"], errors="coerce").fillna(25.0).astype("float64"), heavy_day_cap).astype("float32")
lo, hi = expand_bounds_around_anchor(extension["temp_lower_bound"], extension["temp_upper_bound"], extension["temp_synthetic_center_internal"], 3.0)
extension["temp_fcst_min"] = np.clip(extension["temp_fcst_mean"].astype("float64") - 1.28 * extension["temp_fcst_std"].astype("float64"), lo, hi).astype("float32")
extension["temp_fcst_max"] = np.clip(extension["temp_fcst_mean"].astype("float64") + 1.28 * extension["temp_fcst_std"].astype("float64"), lo, hi).astype("float32")
extension["apparent_temp_fcst_mean"] = apparent_temperature(extension["temp_fcst_mean"], extension["humid_fcst_mean"], extension["wind_fcst_mean"]).astype("float32")
extension["forecast_source"] = SYNTHETIC_FORECAST_SOURCE
extension["forecast_support_level"] = "supported_synthetic_h3_h10"
extension["anchor_horizon"] = np.int8(-1)
extension["max_supported_horizon"] = np.int8(PRIMARY_SUPPORTED_MAX_HORIZON)
extension["calibration_source"] = "realised_latent_weather_climatology_drift_plus_hierarchical_deterministic_error_ensemble_historical_01d"
extension["uncertainty_source"] = "h2_error_growth_ensemble_spread_historical_bounds_pressure_regime_internal"
extension["synthetic_anchor_type"] = "realised_target_day_latent_weather_with_climatology_drift_center"
extension["synthetic_error_method"] = "deterministic_seeded_climatology_drift_horizon_variable_season_regime_calibrated_error"
for col in ["hours_in_window", "expected_hours_in_window", "coverage_share", "is_partial_window", "is_full_window"]:
    extension[col] = np.nan
synthetic_truth_audit = extension[[
    "aggregation_window",
    "horizon_days",
    "AvdelingID",
    "target_date",
    "pressure_regime_internal",
    "temp_obs_mean",
    "temp_fcst_mean",
    "precip_obs_sum",
    "precip_fcst_sum",
    "wind_obs_mean",
    "wind_fcst_mean",
    "humid_obs_mean",
    "humid_fcst_mean",
    "cloud_obs_mean",
    "cloud_fcst_mean",
    "precip_wet_prob",
]].copy()


In [ ]:
COMMON_OUTPUT_COLUMNS = [
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "target_date",
    "horizon_days",
    "AvdelingID",
    "aggregation_window",
    "forecast_source", "forecast_support_level", "anchor_horizon", "max_supported_horizon",
    "h2_anchor_weight", "climatology_weight", "uncertainty_scale", "calibration_source", "uncertainty_source",
    "calibration_fallback_level", "uncertainty_fallback_level", "climatology_fallback_level",
    "climatology_source", "climatology_n_obs", "climatology_expected_obs",
    "climatology_min_year", "climatology_max_year", "climatology_year_span", "climatology_coverage_complete",
    "synthetic_anchor_type", "synthetic_error_method",
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max", "precip_fcst_sum", "wind_fcst_mean", "wind_fcst_max",
    "humid_fcst_mean", "cloud_fcst_mean", "apparent_temp_fcst_mean",
    "temp_fcst_std",
    "precip_wet_prob",
    "precip_amount_uncertainty",
    "wind_fcst_std",
    "humid_fcst_std",
    "cloud_fcst_std",
    "hours_in_window", "expected_hours_in_window", "coverage_share", "is_partial_window", "is_full_window",
    "temp_h2_anchor_weight",
    "precip_h2_anchor_weight",
    "wind_h2_anchor_weight",
    "humid_h2_anchor_weight",
    "cloud_h2_anchor_weight",
]

for col in COMMON_OUTPUT_COLUMNS:
    if col not in det_operational.columns:
        det_operational[col] = np.nan
    if col not in extension.columns:
        extension[col] = np.nan

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
        category=FutureWarning,
    )
    operational = pd.concat(
        [det_operational[COMMON_OUTPUT_COLUMNS], extension[COMMON_OUTPUT_COLUMNS]],
        ignore_index=True,
    )
operational["origin_date"] = pd.to_datetime(operational["origin_date"]).dt.normalize()
operational["target_date"] = pd.to_datetime(operational["target_date"]).dt.normalize()
operational["origin_datetime_utc"] = pd.to_datetime(operational["origin_datetime_utc"], utc=True)
operational["origin_hour"] = operational["origin_hour"].astype("int16")
operational["horizon_days"] = operational["horizon_days"].astype("int16")
operational["AvdelingID"] = operational["AvdelingID"].astype("int64")
operational["aggregation_window"] = pd.Categorical(operational["aggregation_window"], categories=WINDOW_ORDER)

float_cols = [
    "h2_anchor_weight", "climatology_weight", "uncertainty_scale",
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max", "precip_fcst_sum", "wind_fcst_mean", "wind_fcst_max",
    "humid_fcst_mean", "cloud_fcst_mean", "apparent_temp_fcst_mean", "temp_fcst_std", "precip_wet_prob",
    "precip_amount_uncertainty", "wind_fcst_std", "humid_fcst_std", "cloud_fcst_std", "coverage_share",
    "temp_h2_anchor_weight",
    "precip_h2_anchor_weight",
    "wind_h2_anchor_weight",
    "humid_h2_anchor_weight",
    "cloud_h2_anchor_weight",
]
for col in float_cols:
    operational[col] = operational[col].astype("float32")
log_state("after assemble operational (det + synthetic extension)", ("operational", operational))


In [ ]:
# Add calibrated interval features; realised weather enters only through grouped empirical bounds.
# Wet-amount distributions also come from grouped empirical bounds.
interval_work = add_season_month(operational, "target_date")
interval_work = attach_hierarchical_values(
    interval_work,
    BOUNDS_FALLBACK_SPECS,
    BOUNDS_VALUE_COLS,
    "interval_bound_fallback_level",
    BOUNDS_DEFAULTS,
)
for col in BOUNDS_VALUE_COLS:
    interval_work[col] = (
        pd.to_numeric(interval_work[col], errors="coerce")
        .fillna(BOUNDS_DEFAULTS[col])
        .astype("float64")
    )

interval_work = add_bounded_symmetric_interval(
    interval_work,
    "temp_fcst_mean",
    "temp_fcst_std",
    "temp_fcst",
    lower=interval_work["temp_lower_bound"],
    upper=interval_work["temp_upper_bound"],
)
interval_work = add_bounded_symmetric_interval(
    interval_work,
    "wind_fcst_mean",
    "wind_fcst_std",
    "wind_fcst",
    lower=np.zeros(len(interval_work), dtype="float64"),
    upper=interval_work["wind_upper_bound"],
)
interval_work = add_bounded_symmetric_interval(
    interval_work,
    "humid_fcst_mean",
    "humid_fcst_std",
    "humid_fcst",
    lower=np.zeros(len(interval_work), dtype="float64"),
    upper=np.full(len(interval_work), 100.0, dtype="float64"),
)
interval_work = add_bounded_symmetric_interval(
    interval_work,
    "cloud_fcst_mean",
    "cloud_fcst_std",
    "cloud_fcst",
    lower=np.zeros(len(interval_work), dtype="float64"),
    upper=np.full(len(interval_work), 100.0, dtype="float64"),
)
interval_work = add_precipitation_interval(interval_work)
interval_work = add_cloud_regime_probabilities(interval_work)

NEW_INTERVAL_COLUMNS = [
    "temp_fcst_p10", "temp_fcst_p90", "temp_fcst_interval_width",
    "wind_fcst_p10", "wind_fcst_p90", "wind_fcst_interval_width",
    "humid_fcst_p10",
    "humid_fcst_p90",
    "humid_fcst_interval_width",
    "cloud_fcst_p10", "cloud_fcst_p90", "cloud_fcst_interval_width",
    "precip_fcst_p10", "precip_fcst_p90", "precip_fcst_interval_width", "precip_wet_amount_p90",
    "cloud_open_prob", "cloud_partly_prob", "cloud_overcast_prob",
]
for col in NEW_INTERVAL_COLUMNS:
    operational[col] = interval_work[col].astype("float32")


In [ ]:
# Operational output must not expose realised target-day weather or realised forecast errors as features.
# climatology_n_obs and climatology_expected_obs are historical coverage metadata, not target-day leakage.

ALLOWED_OBS_METADATA_COLS = {
    "climatology_n_obs",
    "climatology_expected_obs",
}

realised_like_cols = [
    c for c in operational.columns
    if (
        (
            c.endswith("_obs")
            or "_obs_" in c
            or c.endswith("_error")
            or "forecast_error" in c
        )
        and c not in ALLOWED_OBS_METADATA_COLS
    )
]

pressure_like_cols = [c for c in operational.columns if "pressure" in c.lower()]

if realised_like_cols:
    raise AssertionError(f"Operational output contains diagnostics/realised columns: {realised_like_cols}")

if pressure_like_cols:
    raise AssertionError(f"Operational output exposes pressure columns: {pressure_like_cols}")
key_cols = ["origin_date", "origin_hour", "target_date", "horizon_days", "AvdelingID", "aggregation_window"]
duplicate_count = int(operational.duplicated(key_cols).sum())
if duplicate_count:
    raise AssertionError(f"Duplicate operational forecast keys: {duplicate_count}")

if operational["precip_fcst_sum"].lt(-1e-8).any():
    raise AssertionError("Negative precipitation found in operational output")
if operational["wind_fcst_mean"].lt(-1e-8).any() or operational["wind_fcst_max"].lt(-1e-8).any():
    raise AssertionError("Negative wind found in operational output")
if not operational["humid_fcst_mean"].between(0, 100).all():
    raise AssertionError("Humidity outside [0, 100]")
if not operational["cloud_fcst_mean"].between(0, 100).all():
    raise AssertionError("Cloud cover outside [0, 100]")
for col in [
    "temp_fcst_std",
    "precip_amount_uncertainty",
    "wind_fcst_std",
    "humid_fcst_std",
    "cloud_fcst_std",
]:
    if operational[col].dropna().lt(-1e-8).any():
        raise AssertionError(f"Negative uncertainty in {col}")

CORE_OPERATIONAL_COLUMNS = [
    "temp_fcst_mean", "precip_fcst_sum", "wind_fcst_mean", "humid_fcst_mean", "cloud_fcst_mean",
    "apparent_temp_fcst_mean", "temp_fcst_std", "precip_wet_prob", "precip_amount_uncertainty",
    "wind_fcst_std", "humid_fcst_std", "cloud_fcst_std", "h2_anchor_weight", "climatology_weight",
    "uncertainty_scale",
    "forecast_source",
    "forecast_support_level",
    "calibration_source",
    "uncertainty_source",
    *NEW_INTERVAL_COLUMNS,
]
core_missing = operational[CORE_OPERATIONAL_COLUMNS].isna().sum()
core_missing = core_missing[core_missing.gt(0)]
if not core_missing.empty:
    raise AssertionError(f"Core operational columns contain missing values: {core_missing.to_dict()}")

horizon_values = sorted(operational["horizon_days"].unique().tolist())
if horizon_values != OUTPUT_HORIZONS:
    raise AssertionError(f"Unexpected horizons {horizon_values}; expected {OUTPUT_HORIZONS}")
if sorted(operational["aggregation_window"].dropna().unique().astype(str).tolist()) != WINDOW_ORDER:
    raise AssertionError("Not all expected weather windows are present")

continuous_interval_specs = [
    ("temp", "temp_fcst_mean", "temp_fcst_p10", "temp_fcst_p90", "temp_fcst_interval_width"),
    ("wind", "wind_fcst_mean", "wind_fcst_p10", "wind_fcst_p90", "wind_fcst_interval_width"),
    ("humid", "humid_fcst_mean", "humid_fcst_p10", "humid_fcst_p90", "humid_fcst_interval_width"),
    ("cloud", "cloud_fcst_mean", "cloud_fcst_p10", "cloud_fcst_p90", "cloud_fcst_interval_width"),
]
interval_violation_rows = []
for variable, mean_col, p10_col, p90_col, width_col in continuous_interval_specs:
    lower_violation = int((operational[p10_col] > operational[mean_col] + 1e-6).sum())
    upper_violation = int((operational[p90_col] < operational[mean_col] - 1e-6).sum())
    width_violation = int((operational[width_col] < -1e-8).sum())
    interval_violation_rows.extend([
        {"check": f"{variable}_p10_le_mean", "violation_count": lower_violation},
        {"check": f"{variable}_mean_le_p90", "violation_count": upper_violation},
        {"check": f"{variable}_interval_width_nonnegative", "violation_count": width_violation},
    ])
    if lower_violation or upper_violation or width_violation:
        raise AssertionError(f"Interval ordering failed for {variable}")

if operational[["wind_fcst_p10", "wind_fcst_p90"]].lt(-1e-8).any().any():
    raise AssertionError("Wind interval contains negative values")
if not (
    operational[["humid_fcst_p10", "humid_fcst_p90", "cloud_fcst_p10", "cloud_fcst_p90"]]
    .apply(lambda s: s.between(0, 100).all())
    .all()
):
    raise AssertionError("Humidity/cloud interval outside [0, 100]")
if operational["precip_fcst_p90"].lt(-1e-8).any():
    raise AssertionError("Negative precipitation p90 found")
if operational["precip_fcst_interval_width"].lt(-1e-8).any():
    raise AssertionError("Negative precipitation interval width found")

cloud_prob_cols = ["cloud_open_prob", "cloud_partly_prob", "cloud_overcast_prob"]
if operational[cloud_prob_cols].isna().any().any():
    raise AssertionError("Missing cloud regime probabilities")
if not operational[cloud_prob_cols].apply(lambda s: s.between(0, 1).all()).all():
    raise AssertionError("Cloud regime probabilities outside [0, 1]")
cloud_prob_sum = operational[cloud_prob_cols].sum(axis=1)
cloud_prob_sum_max_abs_error = float(np.abs(cloud_prob_sum - 1.0).max())
if cloud_prob_sum_max_abs_error > 1e-4:
    raise AssertionError(f"Cloud regime probabilities do not sum to 1; max error={cloud_prob_sum_max_abs_error}")


In [ ]:
source_composition = (
    operational.groupby(["aggregation_window", "horizon_days", "forecast_source", "forecast_support_level"], observed=True)
    .size().rename("rows").reset_index()
)
source_composition.to_csv(SOURCE_COMPOSITION_PATH, index=False)

alpha_schedule = pd.DataFrame({"horizon_days": SYNTHETIC_HORIZONS})
alpha_schedule["alpha_realised_weight"] = climatology_alpha_for_horizon(alpha_schedule["horizon_days"]).astype("float32")
alpha_schedule["climatology_weight"] = (1.0 - alpha_schedule["alpha_realised_weight"].astype("float64")).astype("float32")
alpha_schedule["alpha_at_h10"] = float(CLIMATOLOGY_DRIFT_ALPHA_H10)
alpha_schedule["lambda"] = float(-np.log(float(CLIMATOLOGY_DRIFT_ALPHA_H10)) / max(PRIMARY_SUPPORTED_MAX_HORIZON - 2, 1))
alpha_schedule.to_csv(CLIMATOLOGY_DRIFT_ALPHA_SCHEDULE_PATH, index=False)

calibration_weights = (
    operational[[
        "horizon_days", "aggregation_window", "forecast_source", "forecast_support_level", "h2_anchor_weight",
        "climatology_weight", "uncertainty_scale", "temp_h2_anchor_weight", "precip_h2_anchor_weight",
        "wind_h2_anchor_weight", "humid_h2_anchor_weight", "cloud_h2_anchor_weight", "calibration_source",
        "calibration_fallback_level", "uncertainty_fallback_level", "climatology_fallback_level",
        "synthetic_anchor_type", "synthetic_error_method",
    ]]
    .drop_duplicates()
    .sort_values(["aggregation_window", "horizon_days"])
)
calibration_weights.to_csv(CALIBRATION_WEIGHT_PATH, index=False)

coverage_summary = (
    operational.groupby(["aggregation_window", "horizon_days", "forecast_source", "forecast_support_level"], observed=True)
    .agg(
        rows=("AvdelingID", "size"),
        stores=("AvdelingID", "nunique"),
        origin_date_min=("origin_date", "min"),
        origin_date_max=("origin_date", "max"),
        target_date_min=("target_date", "min"),
        target_date_max=("target_date", "max"),
        mean_deterministic_coverage=("coverage_share", "mean"),
    )
    .reset_index()
)
coverage_summary.to_csv(COVERAGE_SUMMARY_PATH, index=False)

interval_summary_rows = []
for variable, mean_col, p10_col, p90_col, width_col in continuous_interval_specs:
    tmp = (
        operational.groupby(["aggregation_window", "horizon_days"], observed=True)
        .agg(
            rows=(mean_col, "size"),
            point_mean=(mean_col, "mean"),
            p10_mean=(p10_col, "mean"),
            p90_mean=(p90_col, "mean"),
            interval_width_mean=(width_col, "mean"),
            interval_width_p90=(width_col, lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.90))),
        )
        .reset_index()
    )
    tmp.insert(0, "variable", variable)
    interval_summary_rows.append(tmp)
precip_interval_summary = (
    operational.groupby(["aggregation_window", "horizon_days"], observed=True)
    .agg(
        rows=("precip_fcst_sum", "size"),
        point_mean=("precip_fcst_sum", "mean"),
        p10_mean=("precip_fcst_p10", "mean"),
        p90_mean=("precip_fcst_p90", "mean"),
        interval_width_mean=("precip_fcst_interval_width", "mean"),
        interval_width_p90=("precip_fcst_interval_width", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.90))),
    )
    .reset_index()
)
precip_interval_summary.insert(0, "variable", "precip")
interval_feature_summary = pd.concat([*interval_summary_rows, precip_interval_summary], ignore_index=True)
interval_feature_summary.to_csv(INTERVAL_FEATURE_SUMMARY_PATH, index=False)

bounds_check_summary = pd.DataFrame([
    *interval_violation_rows,
    {"check": "wind_interval_nonnegative", "violation_count": int(operational[["wind_fcst_p10", "wind_fcst_p90"]].lt(-1e-8).sum().sum())},
    {"check": "humid_interval_bounds", "violation_count": int((~operational[["humid_fcst_p10", "humid_fcst_p90"]].apply(lambda s: s.between(0, 100))).sum().sum())},
    {"check": "cloud_interval_bounds", "violation_count": int((~operational[["cloud_fcst_p10", "cloud_fcst_p90"]].apply(lambda s: s.between(0, 100))).sum().sum())},
    {"check": "precip_p90_nonnegative", "violation_count": int(operational["precip_fcst_p90"].lt(-1e-8).sum())},
    {"check": "cloud_probability_sum_max_abs_error_x1e6", "violation_count": int(round(cloud_prob_sum_max_abs_error * 1_000_000))},
])
bounds_check_summary.to_csv(BOUNDS_CHECK_SUMMARY_PATH, index=False)

precip_bucketed = operational.copy()
precip_bucketed["wet_prob_bucket"] = pd.cut(
    precip_bucketed["precip_wet_prob"],
    bins=[-0.001, 0.10, 0.25, 0.50, 0.75, 1.0001],
    labels=["0.00-0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-1.00"],
)
precip_p90_sanity = (
    precip_bucketed.groupby(["aggregation_window", "horizon_days", "wet_prob_bucket"], observed=True)
    .agg(
        rows=("precip_fcst_sum", "size"),
        precip_fcst_sum_mean=("precip_fcst_sum", "mean"),
        precip_wet_prob_mean=("precip_wet_prob", "mean"),
        precip_amount_uncertainty_mean=("precip_amount_uncertainty", "mean"),
        precip_fcst_p90_mean=("precip_fcst_p90", "mean"),
        precip_fcst_p90_p90=("precip_fcst_p90", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.90))),
    )
    .reset_index()
)
precip_p90_sanity.to_csv(PRECIP_P90_SANITY_PATH, index=False)

cloud_regime_summary = (
    operational.groupby(["aggregation_window", "horizon_days"], observed=True)
    .agg(
        rows=("cloud_fcst_mean", "size"),
        cloud_mean=("cloud_fcst_mean", "mean"),
        cloud_std_mean=("cloud_fcst_std", "mean"),
        cloud_open_prob_mean=("cloud_open_prob", "mean"),
        cloud_partly_prob_mean=("cloud_partly_prob", "mean"),
        cloud_overcast_prob_mean=("cloud_overcast_prob", "mean"),
        cloud_prob_sum_mean=("cloud_open_prob", lambda s: float(cloud_prob_sum.loc[s.index].mean())),
    )
    .reset_index()
)
cloud_regime_summary.to_csv(CLOUD_REGIME_SUMMARY_PATH, index=False)
synthetic_method_summary = operational.groupby(["aggregation_window", "horizon_days", "forecast_source", "forecast_support_level", "synthetic_anchor_type", "synthetic_error_method"], observed=True).size().rename("rows").reset_index()
synthetic_method_summary.to_csv(SYNTHETIC_METHOD_SUMMARY_PATH, index=False)
pressure_regime_audit = extension.groupby(["aggregation_window", "horizon_days", "pressure_regime_internal"], observed=True).agg(rows=("AvdelingID", "size"), precip_wet_prob_mean=("precip_wet_prob", "mean"), precip_fcst_sum_mean=("precip_fcst_sum", "mean"), precip_truth_sum_mean=("precip_obs_sum", "mean"), wind_std_mean=("wind_fcst_std", "mean"), cloud_std_mean=("cloud_fcst_std", "mean")).reset_index()
pressure_regime_audit.to_csv(PRESSURE_REGIME_AUDIT_PATH, index=False)
historical_usage_audit = pd.concat([
    operational.groupby(["aggregation_window", "climatology_fallback_level"], observed=True).size().rename("rows").reset_index().assign(audit="historical_distribution_fallback"),
    interval_work.groupby(["aggregation_window", "interval_bound_fallback_level"], observed=True).size().rename("rows").reset_index().rename(columns={"interval_bound_fallback_level": "climatology_fallback_level"}).assign(audit="empirical_bound_fallback"),
], ignore_index=True, sort=False)
historical_usage_audit.to_csv(HISTORICAL_USAGE_AUDIT_PATH, index=False)
climatology_drift_source_coverage = (
    operational.loc[operational["horizon_days"].between(3, 10)]
    .groupby(["aggregation_window", "horizon_days", "climatology_fallback_level"], observed=True)
    .agg(rows=("AvdelingID", "size"), climatology_weight_mean=("climatology_weight", "mean"), uncertainty_scale_mean=("uncertainty_scale", "mean"))
    .reset_index()
)
climatology_drift_source_coverage["climatology_source_period"] = np.select(
    [
        climatology_drift_source_coverage["climatology_fallback_level"].astype(str).eq("era5_store_same_calendar_day_window"),
        climatology_drift_source_coverage["climatology_fallback_level"].astype(str).str.startswith("era5_"),
        climatology_drift_source_coverage["aggregation_window"].astype(str).eq(SELECTED_OPERATIONAL_WEATHER_WINDOW),
    ],
    [
        "era5_same_calendar_day_2000_2019",
        "era5_legacy_month_or_season_fallback_pre_2022",
        hist_trade_0819_source_label,
    ],
    default="01a_realised_windows_available_history_for_non_main_windows",
)
climatology_drift_source_coverage.to_csv(CLIMATOLOGY_DRIFT_SOURCE_COVERAGE_PATH, index=False)

# ERA5 climatology audits.
ERA5_CLIMATOLOGY_USAGE_PATH       = OUTPUT_DIR / "era5_climatology_usage_summary.csv"
ERA5_CLIMATOLOGY_FALLBACK_PATH    = OUTPUT_DIR / "era5_climatology_fallback_summary.csv"
ERA5_CLIMATOLOGY_LEAKAGE_PATH     = OUTPUT_DIR / "era5_climatology_leakage_check.csv"
ERA5_CLIMATOLOGY_VALUE_RANGE_PATH = OUTPUT_DIR / "era5_climatology_value_range_check.csv"
ERA5_SAME_DAY_VALIDATION_PATH     = OUTPUT_DIR / "era5_same_calendar_day_climatology_validation_summary.csv"

_op_h3plus = operational.loc[operational["horizon_days"].ge(3)].copy()
era5_usage = (
    _op_h3plus.groupby(["aggregation_window", "climatology_fallback_level"], observed=True)
    .agg(rows=("AvdelingID", "size"),
         climatology_weight_mean=("climatology_weight", "mean"))
    .reset_index()
)
era5_usage["uses_era5"] = era5_usage["climatology_fallback_level"].astype(str).str.startswith("era5_")
era5_usage["uses_realised_fallback"] = era5_usage["climatology_fallback_level"].astype(str).str.startswith("realised_")
era5_usage.to_csv(ERA5_CLIMATOLOGY_USAGE_PATH, index=False)

_same_day_rows = int(_op_h3plus["climatology_fallback_level"].astype(str).eq("era5_store_same_calendar_day_window").sum())
_same_day_n20_rows = int((_op_h3plus["climatology_fallback_level"].astype(str).eq("era5_store_same_calendar_day_window") & pd.to_numeric(_op_h3plus.get("climatology_n_obs", 0), errors="coerce").eq(20)).sum())
_fallback_away_rows = int((_op_h3plus["climatology_fallback_level"].astype(str) != "era5_store_same_calendar_day_window").sum())
same_day_validation_summary = pd.concat([
    pd.DataFrame([
        {"summary": "h3_plus_rows", "value": int(len(_op_h3plus))},
        {"summary": "rows_same_calendar_day", "value": _same_day_rows},
        {"summary": "rows_same_calendar_day_n_obs_20", "value": _same_day_n20_rows},
        {"summary": "rows_fallback_away_from_same_calendar_day", "value": _fallback_away_rows},
    ]),
    _op_h3plus.groupby("aggregation_window", observed=True).size().rename("value").reset_index().assign(summary="rows_by_aggregation_window").rename(columns={"aggregation_window": "detail"}),
], ignore_index=True, sort=False)
same_day_validation_summary.to_csv(ERA5_SAME_DAY_VALIDATION_PATH, index=False)

# Same-calendar-day climatology is the required primary h=3-h=10 centre.
if _same_day_rows == 0:
    raise RuntimeError(
        "No h=3..h=10 operational rows use the 'era5_store_same_calendar_day_window' climatology. "
        "Same-calendar-day climatology is required as the primary medium-range centre; refusing to "
        "write an operational file that silently relies on monthly/seasonal or realised fallbacks. "
        "Check the same-day location-key resolution (ERA5 cell) and the climatology fallback specs."
    )

_total_h3plus = len(_op_h3plus)
_era5_rows = int(_op_h3plus["climatology_fallback_level"].astype(str).str.startswith("era5_").sum())
_realised_rows = int(_op_h3plus["climatology_fallback_level"].astype(str).str.startswith("realised_").sum())
_default_rows = int(_op_h3plus["climatology_fallback_level"].astype(str).eq("global_default_floor").sum())
era5_fallback_summary = pd.DataFrame([
    {"summary": "h3_plus_total_rows",        "value": int(_total_h3plus)},
    {"summary": "h3_plus_rows_era5",         "value": _era5_rows},
    {"summary": "h3_plus_rows_realised_emergency_fallback", "value": _realised_rows},
    {"summary": "h3_plus_rows_default_floor","value": _default_rows},
    {"summary": "h3_plus_share_era5",        "value": float(_era5_rows / max(_total_h3plus, 1))},
    {"summary": "h3_plus_share_realised",    "value": float(_realised_rows / max(_total_h3plus, 1))},
    {"summary": "h3_plus_share_default",     "value": float(_default_rows / max(_total_h3plus, 1))},
])
era5_fallback_summary.to_csv(ERA5_CLIMATOLOGY_FALLBACK_PATH, index=False)

_era5_max_src = pd.to_datetime(era5_clim_params["max_source_date"]).max()
era5_leakage = pd.DataFrame([
    {"check": "era5_same_calendar_day_max_year",
     "value": str(int(_same_day_max_year) if pd.notna(_same_day_max_year) else None),
     "boundary": str(ERA5_SAME_DAY_YEAR_END),
     "leakage_safe": bool(_same_day_max_year <= ERA5_SAME_DAY_YEAR_END)},
    {"check": "era5_max_source_date",
     "value": str(_era5_max_src.date() if pd.notna(_era5_max_src) else None),
     "boundary": str(ERA5_MAX_SOURCE_DATE.date()),
     "leakage_safe": bool(_era5_max_src <= ERA5_MAX_SOURCE_DATE)},
    {"check": "era5_param_groups",        "value": int(len(era5_clim_params)), "boundary": "", "leakage_safe": True},
    {"check": "era5_param_avdelings",     "value": int(era5_clim_internal["AvdelingID"].nunique()) if "AvdelingID" in era5_clim_internal.columns else 0, "boundary": 46, "leakage_safe": True},
    {"check": "h3_plus_rows_after_2021",  "value": int((pd.to_datetime(_op_h3plus["target_date"]) > ERA5_MAX_SOURCE_DATE).sum()),
     "boundary": "any (target_date can be in 2022+)", "leakage_safe": True},
])
era5_leakage.to_csv(ERA5_CLIMATOLOGY_LEAKAGE_PATH, index=False)

_era5_value_range_rows = []
for var, col, low, high in [
    ("temp",  "temp_clim",  -40.0, 40.0),
    ("wind",  "wind_clim",   0.0, 50.0),
    ("humid", "humid_clim",  0.0, 100.0),
    ("cloud", "cloud_clim",  0.0, 100.0),
    ("precip","precip_clim", 0.0, 300.0),
    ("precip_wet_prob","precip_wet_prob_clim", 0.0, 1.0),
    ("precip_wet_amount","precip_wet_amount_clim", 0.0, 200.0),
]:
    if col in era5_clim_internal.columns:
        s = pd.to_numeric(era5_clim_internal[col], errors="coerce").dropna()
        _era5_value_range_rows.append({
            "variable": var, "column": col,
            "n": int(len(s)),
            "min": float(s.min()) if len(s) else None,
            "max": float(s.max()) if len(s) else None,
            "expected_min": low, "expected_max": high,
            "in_range_share": float(s.between(low, high).mean()) if len(s) else None,
        })
pd.DataFrame(_era5_value_range_rows).to_csv(ERA5_CLIMATOLOGY_VALUE_RANGE_PATH, index=False)
point_interval_audit_rows = []
for variable, truth_col, fcst_col, p10_col, p90_col in [("temp", "temp_obs_mean", "temp_fcst_mean", "temp_fcst_p10", "temp_fcst_p90"), ("wind", "wind_obs_mean", "wind_fcst_mean", "wind_fcst_p10", "wind_fcst_p90"), ("humid", "humid_obs_mean", "humid_fcst_mean", "humid_fcst_p10", "humid_fcst_p90"), ("cloud", "cloud_obs_mean", "cloud_fcst_mean", "cloud_fcst_p10", "cloud_fcst_p90"), ("precip", "precip_obs_sum", "precip_fcst_sum", "precip_fcst_p10", "precip_fcst_p90")]:
    tmp = operational.loc[operational["horizon_days"].between(3, 10), FORECAST_KEY_COLS + [p10_col, p90_col]].merge(synthetic_truth_audit[["aggregation_window", "horizon_days", "AvdelingID", "target_date", truth_col, fcst_col]], on=["aggregation_window", "horizon_days", "AvdelingID", "target_date"], how="inner")
    tmp["error"] = pd.to_numeric(tmp[fcst_col], errors="coerce") - pd.to_numeric(tmp[truth_col], errors="coerce")
    grouped = tmp.groupby(["aggregation_window", "horizon_days"], observed=True).agg(rows=("error", "size"), truth_mean=(truth_col, "mean"), forecast_mean=(fcst_col, "mean"), error_mean=("error", "mean"), error_rmse=("error", lambda s: float(np.sqrt(np.nanmean(np.square(pd.to_numeric(s, errors="coerce"))))),), p10_mean=(p10_col, "mean"), p90_mean=(p90_col, "mean")).reset_index()
    grouped.insert(0, "variable", variable)
    point_interval_audit_rows.append(grouped)
synthetic_point_interval_audit = pd.concat(point_interval_audit_rows, ignore_index=True)
synthetic_point_interval_audit.to_csv(SYNTHETIC_POINT_INTERVAL_AUDIT_PATH, index=False)

validation_checks = pd.DataFrame([
    {"check": "row_count", "value": len(operational)},
    {"check": "duplicate_key_count", "value": duplicate_count},
    {"check": "horizons", "value": ",".join(map(str, horizon_values))},
    {"check": "windows", "value": ",".join(WINDOW_ORDER)},
    {"check": "selected_window_present", "value": bool((operational["aggregation_window"].astype(str) == SELECTED_OPERATIONAL_WEATHER_WINDOW).any())},
    {"check": "h0_h1_h2_source", "value": ",".join(sorted(operational.loc[operational["horizon_days"].isin([0, 1, 2]), "forecast_source"].unique()))},
    {"check": "h3_h10_source", "value": ",".join(sorted(operational.loc[operational["horizon_days"].between(3, 10), "forecast_source"].unique()))},
    {"check": "synthetic_rows_dropped_missing_realised_truth", "value": missing_truth_rows},
    {"check": "climatology_drift_alpha_h10", "value": float(CLIMATOLOGY_DRIFT_ALPHA_H10)},
    {"check": "synthetic_h10_climatology_weight_mean", "value": float(operational.loc[operational["horizon_days"].eq(10), "climatology_weight"].mean())},
    {"check": "pressure_columns_in_operational_output", "value": ",".join(pressure_like_cols)},
    {"check": "core_operational_missing_values", "value": int(operational[CORE_OPERATIONAL_COLUMNS].isna().sum().sum())},
    {"check": "new_interval_missing_values", "value": int(operational[NEW_INTERVAL_COLUMNS].isna().sum().sum())},
    {"check": "cloud_probability_sum_max_abs_error", "value": cloud_prob_sum_max_abs_error},
    {"check": "precip_p90_low_wet_prob_mean", "value": float(operational.loc[operational["precip_wet_prob"].le(0.10), "precip_fcst_p90"].mean())},
    {"check": "realised_or_error_columns_in_operational_output", "value": ",".join(realised_like_cols)},
])
validation_checks.to_csv(OUTPUT_CHECKS_PATH, index=False)

log_state("before write operational parquet", ("operational", operational), ("forecast_error_diagnostics", forecast_error_diagnostics))
operational.to_parquet(OPERATIONAL_OUTPUT_PATH, index=False)
print(f"Wrote {OPERATIONAL_OUTPUT_PATH} with {len(operational):,} rows and {operational.shape[1]} columns")
print(f"Wrote {ERROR_DIAGNOSTICS_OUTPUT_PATH} with {len(forecast_error_diagnostics):,} rows")
print("Forecast source composition:")
display(source_composition.head(20))
print("Operational validation checks:")
display(validation_checks)

contract_summary = pd.DataFrame([
    {"item": "main_window", "value": SELECTED_OPERATIONAL_WEATHER_WINDOW},
    {"item": "robustness_window", "value": ROBUSTNESS_WEATHER_WINDOW},
    {"item": "benchmark_window", "value": BENCHMARK_WEATHER_WINDOW},
    {"item": "primary_supported_max_horizon", "value": PRIMARY_SUPPORTED_MAX_HORIZON},
    {"item": "extended_h11_h14_produced", "value": INCLUDE_EXTENDED_LOW_SUPPORT_HORIZONS},
    {"item": "synthetic_h3_h10_method", "value": SYNTHETIC_FORECAST_SOURCE},
    {"item": "synthetic_rows_dropped_missing_realised_truth", "value": missing_truth_rows},
    {"item": "operational_rows", "value": len(operational)},
    {"item": "diagnostic_error_rows", "value": len(forecast_error_diagnostics)},
])
display(contract_summary)


## Manual run and downstream contract

Run this notebook after notebooks 01a, 01b, and 01c have produced and validated their processed weather-window files. A successful run writes:

- `data/processed/weather_forecast_operational_windows.parquet`
- `data/processed/weather_forecast_error_diagnostics_windows.parquet`
- `outputs/synthetic_weather_model/forecast_source_composition.csv`
- `outputs/synthetic_weather_model/synthetic_extension_calibration_summary.csv`
- `outputs/synthetic_weather_model/synthetic_extension_weights_by_horizon.csv`
- `outputs/synthetic_weather_model/weather_forecast_operational_checks.csv`
- `outputs/synthetic_weather_model/interval_feature_summary_by_horizon_window.csv`
- `outputs/synthetic_weather_model/interval_bounds_check_summary.csv`
- `outputs/synthetic_weather_model/precip_p90_sanity_by_horizon_wet_prob_bucket.csv`
- `outputs/synthetic_weather_model/cloud_regime_probability_summary.csv`

Notebooks 05 and 06 should consume `weather_forecast_operational_windows.parquet` as the operational forecast-weather source. `weather_forecast_error_diagnostics_windows.parquet` is for calibration, audit, and thesis appendix use only, and must not be used as an ML feature table.
